# 📘 Zero → Hero MLIR

## *A Practical Tutorial for Fault-Tolerant Compiler Construction*



## 🧭 Notebook Philosophy

> By the end of this notebook, the reader should be able to:

* Understand **why MLIR exists**

* Read and write **custom MLIR dialects**

* Design **fault-tolerant IR abstractions**

* Build **analysis + transformation passes**

* Map **logical IR → hardware-aware IR**

* Reason about **error models at the IR level**


# 🧱 PART 0 — Orientation & Mental Models

### 0.1 What This Notebook Is (and Is Not)

**✅ What This Notebook IS:**

* **Hands-on MLIR tutorial** - You'll write MLIR code, not just read about it
* **Compiler-builder mindset** - Learn to think like someone building compiler infrastructure
* **Fault-tolerance focused** - Everything is viewed through the lens of building fault-tolerant systems
* **Progressive learning** - Builds from zero to advanced concepts
* **Practical examples** - Real-world scenarios and code you can run

**❌ What This Notebook IS NOT:**

* Not a research proposal - This is a tutorial, not an academic paper
* Not a framework comparison - We focus on MLIR, not "MLIR vs X"
* Not a complete MLIR reference - We cover what's needed for fault tolerance
* Not theory-only - Every concept has practical implementation

**🎯 Learning Path:**

This notebook follows a specific progression:

1. **Foundation** (Parts 0-2) → Mental models and core concepts
2. **Building Blocks** (Parts 3-5) → Dialects, types, attributes
3. **Infrastructure** (Parts 6-7) → Verification and passes
4. **Integration** (Parts 8-9) → Lowering and case studies
5. **Application** (Parts 10-11) → Real systems and extensions


### 0.2 Why MLIR for Fault Tolerance?

#### The Classical Compiler Problem

Traditional compilers (like GCC, Clang) are designed with a fundamental assumption:
> **Hardware is reliable. Errors don't exist at the IR level.**

This works for desktop CPUs, but breaks down for:
- **Quantum computers** - Decoherence, gate errors, measurement noise
- **Radiation-hardened systems** - Single-event upsets, bit flips
- **Approximate computing** - Intentional precision trade-offs
- **Unreliable memory** - Flash memory, emerging technologies

**Classical Approach:**
```
Source Code → IR → Assembly → Machine Code
              ↑
         (Perfect abstraction)
```

**Problem:** When hardware is unreliable, hiding faults in the compiler means you **can't reason about errors** until runtime (too late!).

#### Why MLIR Changes Everything

MLIR is **not a single IR** — it's a **framework for building IRs**. This enables:

**1. Multi-Level Abstractions**

You can represent the same computation at multiple levels:

```
High-level: "Compute matrix multiplication"
    ↓
Logical: "Apply quantum gate with error correction"
    ↓
Physical: "Execute CNOT on qubits 3 and 7, then measure stabilizers"
    ↓
Hardware: "Pulse sequence, timing constraints, error rates"
```

Each level has its own **semantic meaning** and **error model**.

**2. Explicit Error Metadata**

Unlike traditional IRs, MLIR lets you attach error information **directly to operations**:

```mlir
// Traditional IR: Can't express errors
%result = add %a, %b

// MLIR: Explicit error model
%result = faulty.add %a, %b {ber = 1e-6, error_model = "bit-flip"}
```

**3. Progressive Lowering**

You can **gradually transform** high-level fault-tolerant code into hardware:

```mlir
// Level 1: Algorithm with redundancy
ft.add %a, %b {redundancy = 3}

// Level 2: Explicit correction blocks
logical.add %a, %b
logical.error_correct %result

// Level 3: Hardware-aware
physical.add %q0, %q1 {gate_error = 0.001}
physical.stabilizer_measure %ancilla
```

**Key Idea:**

> *Fault tolerance is not an afterthought — it is an IR design choice.*

In MLIR, you **design your dialect** to make faults explicit from the beginning. This enables:
- **Compile-time verification** - Catch unsafe transformations early
- **Error-aware optimization** - Optimize for reliability, not just performance
- **Hardware mapping** - Match error models to actual hardware characteristics


# 🧱 PART 1 — Compiler Foundations (Minimal but Essential)

### 1.1 What Is an Intermediate Representation (IR)?

An **Intermediate Representation (IR)** is a data structure that sits between source code and machine code. Think of it as the compiler's "working language."

#### The Compilation Pipeline

```
Source Code (Python/C++) 
    ↓ [Parsing]
Abstract Syntax Tree (AST)
    ↓ [Semantic Analysis]
Intermediate Representation (IR)
    ↓ [Optimization]
    ↓ [Code Generation]
Machine Code (Assembly/Binary)
```

#### AST vs IR vs Machine Code

**Abstract Syntax Tree (AST):**
- **Structure**: Tree-like, mirrors source code syntax
- **Purpose**: Capture program structure and syntax
- **Level**: Very high-level, language-specific
- **Example**: `if (x > 0) { y = 1; }` → nested tree nodes

```python
# AST representation (conceptual)
IfStatement(
    condition=GreaterThan(x, 0),
    then=Assignment(y, 1)
)
```

**Intermediate Representation (IR):**
- **Structure**: Graph-based (SSA form), language-independent
- **Purpose**: Enable analysis and optimization
- **Level**: Medium-level, hardware-agnostic operations
- **Example**: `%1 = icmp sgt %x, 0` → `br %1, label1, label2`

```llvm
; LLVM IR example
%1 = icmp sgt i32 %x, 0
br i1 %1, label %if_true, label %if_false
if_true:
  store i32 1, i32* %y
  br label %end
```

**Machine Code:**
- **Structure**: Linear sequence of instructions
- **Purpose**: Execute on specific hardware
- **Level**: Low-level, hardware-specific
- **Example**: `cmp eax, 0` → `jle .L1` (x86 assembly)

#### Why Single IRs Fail for Complex Systems

Traditional compilers use **one IR** (like LLVM IR). This works well for general-purpose computing, but fails for specialized domains:

**Problem 1: Quantum Computing**

```llvm
; LLVM IR can't express quantum operations
%result = ??? %qubit1, %qubit2  ; What operation?
```

Quantum operations need concepts like:
- Superposition, entanglement
- Error correction codes
- Logical vs physical qubits
- Gate fidelities

**Problem 2: Fault Tolerance**

```llvm
; LLVM IR hides hardware faults
%sum = add i32 %a, %b  ; What if this operation has a bit-error rate?
```

Fault-tolerant systems need:
- Error rates per operation
- Redundancy levels
- Error correction strategies
- Reliability budgets

**Problem 3: Domain-Specific Abstractions**

Different domains need different abstractions:
- **Neural networks**: Tensors, convolution operations, dataflow
- **Signal processing**: Filters, transforms, real-time constraints
- **Database queries**: Relational algebra, query plans
- **Graphics**: Shaders, pipelines, textures

**One IR cannot represent all of these elegantly.**

### 1.2 Why Multi-Level IRs Matter

MLIR solves this by allowing **multiple IRs** (called "dialects") that can coexist and progressively transform into each other.

#### Separation of Concerns

**1. Algorithm Level**
- **Purpose**: Express *what* to compute
- **Concerns**: Correctness, semantics
- **Example**: `matrix_multiply(A, B)`

**2. Error Correction Level**
- **Purpose**: Express *how to protect* the computation
- **Concerns**: Redundancy, error detection/correction
- **Example**: `triple_redundant(matrix_multiply(A, B))`

**3. Scheduling Level**
- **Purpose**: Express *when* operations execute
- **Concerns**: Dependencies, parallelism, latency
- **Example**: Schedule operations respecting dependencies

**4. Hardware Constraints Level**
- **Purpose**: Express *hardware limitations*
- **Concerns**: Gate fidelities, connectivity, timing
- **Example**: Map to specific qubits with error rates

#### The MLIR Advantage

```
Algorithm Dialect (high-level)
    ↓
Fault-Tolerant Dialect (adds error protection)
    ↓
Logical Dialect (error-corrected operations)
    ↓
Physical Dialect (hardware-mapped)
    ↓
LLVM Dialect (executable code)
```

Each level can **reason about its own concerns** without mixing them together.

> ⚠️ This section prepares the reader to *feel* why MLIR exists. The next sections will show you *how* to use it.


# 🧱 PART 2 — MLIR Core Concepts (Absolute Zero Level)

### 2.1 What Is MLIR?

**MLIR** (Multi-Level Intermediate Representation) is a compiler infrastructure project that's part of the **LLVM ecosystem**. 

#### Key Facts

- **Built inside the LLVM Project** - Uses LLVM's infrastructure, tools, and philosophy
- **IR *infrastructure*, not a language** - It's a framework for *building* IRs, not a single IR
- **Language-agnostic** - Works with any source language
- **Domain-agnostic** - Works with any application domain (quantum, ML, graphics, etc.)

#### What MLIR Provides

1. **A common foundation** - Shared infrastructure (parsing, printing, verification)
2. **Dialect system** - Way to define custom IRs with their own operations
3. **Lowering framework** - Transform between different IR levels
4. **Pass infrastructure** - Analysis and transformation passes

Think of MLIR as a **"compiler for compilers"** — it helps you build domain-specific compilers.

### 2.2 MLIR Design Pillars

MLIR is built on five core concepts that you need to understand:

#### 1. Dialects

A **dialect** is a collection of related operations, types, and attributes. Each dialect has its own semantic domain.

**Examples:**
- `func` dialect - Function definitions and calls
- `arith` dialect - Arithmetic operations (add, sub, mul, etc.)
- `memref` dialect - Memory references and operations
- `scf` dialect - Structured control flow (loops, conditionals)

You can also define **custom dialects** (like a `faulty` dialect for fault tolerance).

#### 2. Operations

An **operation** (op) is the fundamental unit of computation in MLIR. Every operation has:
- **Name**: e.g., `add`, `func.func`, `faulty.correct`
- **Operands**: Input values
- **Results**: Output values
- **Attributes**: Static metadata (like operation properties)
- **Regions**: Nested operation groups (for control flow)

```mlir
// Operation syntax: op_name %operand1, %operand2 {attribute = value}
%result = arith.addi %a, %b : i32
```

#### 3. Types

**Types** describe the structure and properties of values. MLIR has:
- **Builtin types**: `i32` (32-bit integer), `f64` (64-bit float), `index`
- **Dialect types**: `memref<10x20xf32>` (tensor type), `!quant.qint8`
- **Custom types**: `!ft.i32<ber=1e-9>` (fault-tolerant type you define)

Types can carry **semantic information** beyond just data representation.

#### 4. Attributes

**Attributes** are compile-time constant metadata attached to operations or types. They don't change at runtime.

```mlir
func.func @example(%x: i32) -> i32 attributes {inline_hint} {
  // Attribute: inline_hint tells the compiler to inline this function
  return %x : i32
}
```

Attributes are used for:
- Optimization hints
- Error models
- Configuration parameters
- Static analysis metadata

#### 5. Regions & Blocks

A **region** is a container for operations. A **block** is a sequence of operations with a single entry point.

**Structure:**
```
Operation (with region)
  └── Region
      ├── Block 1
      │   ├── Operation 1
      │   ├── Operation 2
      │   └── ...
      └── Block 2 (optional)
```

**Example:**
```mlir
func.func @example() {
  // This is Block 1
  %x = arith.constant 42 : i32
  %y = arith.constant 10 : i32
  return
}
```

📌 **Mental Model**

```
MLIR = Lego box for IR design
  └── Contains all the pieces and tools

Dialect = A themed Lego set
  └── "City set", "Space set", "Fault-Tolerance set"

Op = A Lego brick
  └── Individual building blocks (add, multiply, correct)

Type = The shape/specification of a brick
  └── "2x2 brick", "wheel", "fault-tolerant integer"

Attribute = Sticker on a brick
  └── Metadata (color, special property)

Region/Block = How bricks are assembled
  └── The structure and organization
```


### 2.3 Your First MLIR Program

Let's examine a simple MLIR program to identify all the core concepts:


In [1]:
# Your first MLIR program
mlir_program = """
module {
  func.func @hello() {
    return
  }
}
"""

print("=" * 60)
print("MLIR Program:")
print("=" * 60)
print(mlir_program)
print("=" * 60)


MLIR Program:

module {
  func.func @hello() {
    return
  }
}



#### Breaking Down the Program

Let's identify each component:

**1. `module { ... }` - The Module**

- The **top-level container** in MLIR
- Every MLIR program starts with a `module`
- Contains all functions, global variables, and other top-level constructs
- Type: `builtin.module` (from the `builtin` dialect)

**2. `func.func @hello()` - The Function Operation**

- **Dialect**: `func` (function dialect)
- **Operation name**: `func.func`
- **Name**: `@hello` (the `@` prefix means "symbol name")
- **Signature**: Takes no arguments `()`, returns nothing (implicitly `void`)

**3. `{ return }` - The Function Body (Region)**

- The `{ ... }` is a **region** attached to the `func.func` operation
- Contains the operations that make up the function body
- This region has exactly **one block** (the default entry block)

**4. The Block**

- Implicitly, there's a **block** inside the region
- Contains the sequence of operations: `return`
- The `return` operation exits the function

**5. `return` - Return Operation**

- **Operation**: `func.return` (from `func` dialect)
- Terminates the function execution
- No operands (this function returns nothing)

#### Visual Structure

```
module {                          ← Module (top-level container)
  func.func @hello() {            ← Operation (func.func)
    return                         ← Operation (func.return)
    └── [Block]                   ← Block (implicit, contains operations)
  }                                ← Region (boundary of function body)
}
```

**Exercise: Identify Components**

Try this more complex example:

```mlir
module {
  func.func @add(%a: i32, %b: i32) -> i32 {
    %result = arith.addi %a, %b : i32
    func.return %result : i32
  }
}
```

**Questions:**
1. How many operations are in the function body?
2. What dialect does `arith.addi` belong to?
3. What are the operands to `arith.addi`?
4. What is the type of `%result`?
5. How many blocks are in the function region? (Hint: just one, the default block)

**Answers:**
1. Two operations: `arith.addi` and `func.return`
2. `arith` dialect (arithmetic operations)
3. `%a` and `%b` (the function arguments)
4. `i32` (32-bit integer)
5. One block (the default entry block containing both operations)


# 🧱 PART 3 — Dialects: The Heart of MLIR

### 3.1 What Is a Dialect?

A **dialect** in MLIR is a **semantic universe** — a collection of related concepts that share the same domain of meaning.

#### Dialect Components

A dialect owns and defines:

**1. Operations (Ops)**
- The computational primitives in this domain
- Example: `func.func`, `func.call`, `func.return` (all from `func` dialect)

**2. Types**
- Data types specific to this domain
- Example: `memref<10x20xf32>` (memory reference type from `memref` dialect)

**3. Attributes**
- Domain-specific metadata
- Example: `{sym_visibility = "private"}` (symbol visibility from `func` dialect)

**4. Verification Rules**
- Rules that ensure operations are used correctly
- Example: A `func.return` must have the correct number/type of return values

#### Why Dialects Matter

Dialects provide **semantic isolation**. Operations from different dialects can coexist in the same program, but each dialect defines its own meaning:

```mlir
module {
  // func dialect: function definition
  func.func @compute(%x: i32) -> i32 {
    // arith dialect: arithmetic operations
    %one = arith.constant 1 : i32
    %result = arith.addi %x, %one : i32
    
    // func dialect: return operation
    func.return %result : i32
  }
}
```

Here, `func` and `arith` dialects work together, but each maintains its own semantic rules.

### 3.2 Standard Dialects Tour (Only What We Need)

MLIR comes with many built-in dialects. We'll focus on the essential ones for fault-tolerant systems:

#### 1. `builtin` Dialect

**Purpose**: Fundamental infrastructure types and operations

**Key Operations:**
- `builtin.module` - Top-level module container
- No explicit ops usually needed (used implicitly)

**Key Types:**
- `i32`, `i64`, `f32`, `f64` - Basic numeric types
- `index` - Platform-dependent integer type
- `none` - Unit type (like `void`)

**Example:**
```mlir
// builtin types are used everywhere
%x: i32 = arith.constant 42 : i32
```

#### 2. `func` Dialect

**Purpose**: Function definitions and calls

**Key Operations:**
- `func.func` - Define a function
- `func.call` - Call a function
- `func.return` - Return from a function

**Example:**
```mlir
func.func @add(%a: i32, %b: i32) -> i32 {
  %sum = arith.addi %a, %b : i32
  func.return %sum : i32
}

func.func @main() {
  %result = func.call @add(%1: i32, %2: i32) : (i32, i32) -> i32
  return
}
```

#### 3. `arith` Dialect

**Purpose**: Arithmetic operations

**Key Operations:**
- `arith.addi`, `arith.subi`, `arith.muli` - Integer arithmetic
- `arith.addf`, `arith.subf`, `arith.mulf` - Float arithmetic
- `arith.constant` - Constant values
- `arith.cmpi`, `arith.cmpf` - Comparisons

**Example:**
```mlir
%x = arith.constant 10 : i32
%y = arith.constant 20 : i32
%sum = arith.addi %x, %y : i32
%is_positive = arith.cmpi sgt, %sum, %zero : i32  // signed greater than
```

#### 4. `memref` Dialect

**Purpose**: Memory references and buffer operations

**Key Operations:**
- `memref.alloc` - Allocate memory
- `memref.load` - Load from memory
- `memref.store` - Store to memory
- `memref.get_global` - Get global memory reference

**Key Types:**
- `memref<d1x...xdnxelement_type>` - Multi-dimensional memory reference

**Example:**
```mlir
%mem = memref.alloc() : memref<10xi32>
memref.store %value, %mem[%index] : memref<10xi32>
%loaded = memref.load %mem[%index] : memref<10xi32>
```

#### 5. `scf` Dialect

**Purpose**: Structured control flow (loops, conditionals)

**Key Operations:**
- `scf.if` - Conditional execution
- `scf.for` - For loops
- `scf.while` - While loops
- `scf.yield` - Yield values from control flow

**Example:**
```mlir
%result = scf.if %condition -> i32 {
  // then branch
  %then_val = arith.constant 1 : i32
  scf.yield %then_val : i32
} else {
  // else branch
  %else_val = arith.constant 0 : i32
  scf.yield %else_val : i32
}

scf.for %i = %c0 to %c10 step %c1 iter_args(%sum = %c0) -> i32 {
  %new_sum = arith.addi %sum, %i : i32
  scf.yield %new_sum : i32
}
```

> ⚠️ We deliberately avoid overloading here. There are many more dialects (tensor, linalg, vector, etc.), but these five give us enough power for fault-tolerant system design.


### 3.3 Designing Your First Custom Dialect

**Goal:** Create a **Fault-Aware Dialect** that exposes hardware faults at the IR level.

#### Design Philosophy

For fault tolerance, we need operations that **explicitly model errors**. This means:

1. **Operations must declare their error characteristics**
2. **Types can encode reliability information**
3. **The dialect should support error correction operations**

#### Example Operations for a `faulty` Dialect

**Core Arithmetic Operations:**

```mlir
// Addition with bit-error rate
%result = faulty.add %a, %b {ber = 1e-6} : (i32, i32) -> i32

// Multiplication with error model
%result = faulty.mul %a, %b {error_model = "bit-flip"} : (i32, i32) -> i32

// Memory load with fault probability
%value = faulty.load %mem[%idx] {fault_prob = 0.001} : memref<100xi32> -> i32
```

**Error Correction Operations:**

```mlir
// Triple modular redundancy (TMR)
%corrected = faulty.tmr %value1, %value2, %value3 : (i32, i32, i32) -> i32

// Error detection
%is_valid = faulty.check %value {parity_bit = %parity} : i32 -> i1

// Error correction code application
%encoded = faulty.encode %data {code = "hamming"} : i32 -> !faulty.encoded<i32>
%decoded = faulty.decode %encoded {code = "hamming"} : !faulty.encoded<i32> -> i32
```

#### Key Design Questions

**1. What faults exist?**

You need to model the **fault model** for your target system:

- **Bit-flip errors**: Random bit inversions (common in radiation environments)
- **Stuck-at faults**: Bits stuck at 0 or 1
- **Transient errors**: Temporary faults that correct themselves
- **Gate errors**: Quantum gate fidelity errors
- **Measurement errors**: Errors in reading values

**Example:**
```mlir
// Different fault types
faulty.add %a, %b {fault_type = "bit-flip", ber = 1e-6}
faulty.mul %a, %b {fault_type = "stuck-at-0", prob = 0.001}
faulty.measure %q {error_rate = 0.01} : !quantum.qubit -> i1
```

**2. Where should faults live?**

Faults can be expressed at different levels:

**Option A: Operation Attributes**
```mlir
// Fault info in operation attributes
faulty.add %a, %b {ber = 1e-6}
```
- ✅ Simple, explicit per-operation
- ❌ Verbose, doesn't compose well

**Option B: Type System**
```mlir
// Fault info in types
%result: !faulty.i32<ber=1e-6> = faulty.add %a, %b
```
- ✅ Composable (type carries error through operations)
- ✅ More elegant
- ❌ More complex type system

**Option C: Hybrid Approach** (Recommended)
```mlir
// Types carry baseline error, operations can override
%a: !faulty.i32<ber=1e-6>
%b: !faulty.i32<ber=1e-6>
%result: !faulty.i32<ber=1e-6> = faulty.add %a, %b {override_ber = 1e-5}
```
- ✅ Best of both worlds
- ✅ Flexible and composable

**3. Compile-time vs Runtime?**

**Compile-time (Static):**
- Error rates known at compile time
- Can optimize for reliability
- Can verify fault tolerance properties
- Example: Gate fidelities in quantum hardware specs

**Runtime (Dynamic):**
- Error rates measured at runtime
- Adaptive error correction
- Runtime monitoring and adjustment
- Example: Real-time error rate measurement

**Hybrid Approach:**
```mlir
// Compile-time error model
faulty.add %a, %b {ber = 1e-6} : i32 -> i32

// Runtime error measurement hook
%actual_ber = faulty.measure_error_rate %op : !faulty.op -> f64

// Adaptive correction based on runtime measurement
%corrected = scf.if %actual_ber > %threshold -> i32 {
  faulty.strong_correction %result : i32 -> i32
} else {
  faulty.weak_correction %result : i32 -> i32
}
```

#### Dialect Design Checklist

When designing your `faulty` dialect, consider:

- [ ] **Operations**: What operations can fail? (add, mul, load, store, etc.)
- [ ] **Types**: How do types encode reliability? (`!faulty.i32<ber=X>`)
- [ ] **Attributes**: What metadata is needed? (error rates, fault models, correction strategies)
- [ ] **Verification**: What rules enforce correctness? (e.g., "can't lower logical ops to physical without error correction")
- [ ] **Lowering**: How does this dialect lower to hardware? (faulty → physical → LLVM)

**Exercise: Design Your Dialect**

Sketch out a `faulty` dialect specification:

1. **List 5 operations** you'd need for a fault-tolerant arithmetic kernel
2. **Define 2 types** that encode fault information
3. **Describe 1 verification rule** that would catch unsafe code
4. **Explain how** operations from `faulty` dialect would lower to `arith` dialect (with error correction)


# 🧱 PART 4 — MLIR Type System (Where Fault Tolerance Begins)

### 4.1 Why Types Matter More Than Ops

In traditional programming languages, types tell you "what shape the data has" (int, float, struct). In fault-tolerant systems, **types must also tell you "how reliable the data is"**.

#### Types Encode Semantic Information

**1. Precision**
- How many bits are actually meaningful?
- Example: `f32` has 32 bits, but with noise, only 24 bits might be reliable

**2. Uncertainty**
- What is the error bound?
- Example: `ft.f32<error=0.01>` means the value has ±1% uncertainty

**3. Reliability**
- What is the probability of a fault?
- Example: `ft.i32<ber=1e-9>` means bit-error rate of 10⁻⁹

#### The Power of Type-Driven Fault Tolerance

When types carry fault information, the compiler can:

**1. Propagate Errors Automatically**
```mlir
// If inputs have error, output inherits it
%a: !ft.i32<ber=1e-6>
%b: !ft.i32<ber=1e-6>
%sum = arith.addi %a, %b : i32
// Compiler knows: %sum also has ber ≈ 1e-6 (or slightly worse)
```

**2. Enforce Correctness at Compile-Time**
```mlir
// Type system prevents unsafe operations
%reliable: !ft.i32<ber=1e-9>
%unreliable: !ft.i32<ber=1e-3>

// This could be an error (type mismatch) or require explicit conversion
%result = arith.addi %reliable, %unreliable : i32
// Should we allow this? What's the resulting type?
```

**3. Guide Optimization**
```mlir
// If type says data is unreliable, maybe add redundancy automatically
%x: !ft.i32<ber=1e-3>  // High error rate
%y: !ft.i32<ber=1e-9>  // Low error rate

// Optimizer might:
// - Add redundancy for %x operations
// - Skip redundancy for %y operations (save resources)
```

#### Types vs Operations: Why Types Win

**Operations are ephemeral** - They execute and disappear.

**Types are persistent** - They flow through the entire computation.

This makes types the **natural place** to encode fault tolerance properties.

### 4.2 Custom Types for Fault Tolerance

MLIR allows you to define **custom types** with **parameters** that encode fault information.

#### Type Syntax

MLIR custom types follow this pattern:
```mlir
!dialect.type<parameter1=value1, parameter2=value2>
```

The `!` prefix indicates a custom type (not a builtin type like `i32`).

#### Example: Fault-Tolerant Integer Type

**Syntax:**
```mlir
!ft.i32<ber=1e-9>
```

**Breakdown:**
- `!ft` - Dialect name prefix (fault-tolerant dialect)
- `i32` - Base type (32-bit integer)
- `<ber=1e-9>` - Type parameter (bit-error rate = 10⁻⁹)

**Semantics:**
- This is a 32-bit integer
- Each bit has probability 10⁻⁹ of flipping
- Operations on this type must account for the error rate

#### Example: Logical vs Physical Qubits

**Logical Qubit (Error-Corrected):**
```mlir
!quantum.qubit<level=logical, code=surface_code>
```

**Physical Qubit (Raw Hardware):**
```mlir
!quantum.qubit<level=physical, gate_error=0.001, meas_error=0.01>
```

**Key Difference:**
- Logical qubits are protected by error correction
- Physical qubits have explicit error rates
- Lowering from logical → physical requires error correction operations

#### More Complex Type Examples

**1. Type with Multiple Parameters:**
```mlir
!ft.i32<ber=1e-6, redundancy=3, correction=hamming>
```

**2. Type with Nested Parameters:**
```mlir
!ft.tensor<10x20x!ft.f32<ber=1e-5>>
```

**3. Type with Reliability Budget:**
```mlir
!ft.i32<reliability=0.999, confidence=0.95>
```

#### Type System Design Decisions

When designing fault-tolerant types, consider:

**1. What Parameters Matter?**
- Error rates? (ber, gate_error, meas_error)
- Redundancy level? (how many copies)
- Correction strategy? (TMR, ECC, surface code)
- Precision/uncertainty? (how many bits are reliable)

**2. Static vs Dynamic?**
- Static: `!ft.i32<ber=1e-6>` (compile-time constant)
- Dynamic: `!ft.i32<ber=%runtime_ber>` (runtime value) - more complex

**3. Type Compatibility?**
- Can `!ft.i32<ber=1e-6>` mix with `!ft.i32<ber=1e-9>`?
- What's the resulting type when operations combine different BERs?

**4. Type Inference Rules?**
```mlir
// If we add two values with different BERs, what's the result?
%a: !ft.i32<ber=1e-6>
%b: !ft.i32<ber=1e-9>
%sum = arith.addi %a, %b : i32
// Result type: !ft.i32<ber=???>  // How do we compute this?
```


In [2]:
# Examples of fault-tolerant types in MLIR

print("=" * 70)
print("Fault-Tolerant Type Examples")
print("=" * 70)

# Basic fault-tolerant integer with bit-error rate
basic_type = "!ft.i32<ber=1e-9>"
print("\n1. Basic Type (Bit-Error Rate):")
print(f"   {basic_type}")

# Type with redundancy information
redundancy_type = "!ft.i32<ber=1e-6, redundancy=3>"
print("\n2. Type with Redundancy:")
print(f"   {redundancy_type}")

# Logical qubit type (error-corrected)
logical_qubit = "!quantum.qubit<level=logical, code=surface_code>"
print("\n3. Logical Qubit (Error-Corrected):")
print(f"   {logical_qubit}")

# Physical qubit type (hardware-level)
physical_qubit = "!quantum.qubit<level=physical, gate_error=0.001>"
print("\n4. Physical Qubit (Hardware-Level):")
print(f"   {physical_qubit}")

# Type with precision information
precision_type = "!ft.f32<error=0.01, reliable_bits=24>"
print("\n5. Type with Precision:")
print(f"   {precision_type}")

# Complex nested type
nested_type = "!ft.tensor<10x20x!ft.f32<ber=1e-5>>"
print("\n6. Nested Type (Tensor of Fault-Tolerant Floats):")
print(f"   {nested_type}")

print("\n" + "=" * 70)


Fault-Tolerant Type Examples

1. Basic Type (Bit-Error Rate):
   !ft.i32<ber=1e-9>

2. Type with Redundancy:
   !ft.i32<ber=1e-6, redundancy=3>

3. Logical Qubit (Error-Corrected):
   !quantum.qubit<level=logical, code=surface_code>

4. Physical Qubit (Hardware-Level):
   !quantum.qubit<level=physical, gate_error=0.001>

5. Type with Precision:
   !ft.f32<error=0.01, reliable_bits=24>

6. Nested Type (Tensor of Fault-Tolerant Floats):
   !ft.tensor<10x20x!ft.f32<ber=1e-5>>



**Exercise: Design a Fault-Tolerant Type**

Design a custom type that captures both **bit-error rate** and **redundancy level**.

#### Requirements

Your type should:
1. Specify the base type (e.g., `i32`, `f64`)
2. Include a bit-error rate parameter (`ber`)
3. Include a redundancy level parameter (how many redundant copies)
4. Follow MLIR custom type syntax: `!dialect.name<params>`

#### Example Design

```mlir
!ft.i32<ber=1e-6, redundancy=3>
```

**Questions to Consider:**

1. **Semantics**: What does `redundancy=3` mean?
   - Does it mean the value is stored as 3 copies?
   - Or does it mean operations use 3-way voting?

2. **Type Compatibility**: If you have two values:
   ```mlir
   %a: !ft.i32<ber=1e-6, redundancy=3>
   %b: !ft.i32<ber=1e-9, redundancy=1>
   ```
   - Can you add them? What's the result type?
   - Should redundancy be "inherited" or "upgraded"?

3. **Type Inference**: What happens when operations combine values?
   ```mlir
   %sum = arith.addi %a, %b : i32
   // Result: !ft.i32<ber=???, redundancy=???>
   ```

4. **Lowering**: How does this type lower to hardware?
   - If `redundancy=3`, does it become 3 physical values?
   - Or does it trigger TMR insertion during lowering?

#### Suggested Solution Sketch

```mlir
// Type definition (conceptual)
!ft.i32<
  ber=1e-6,              // Bit-error rate per copy
  redundancy=3,          // Number of redundant copies
  voting_strategy=majority,  // How to combine copies
  correction=auto        // Auto-insert correction ops
>

// Usage in operations
%a: !ft.i32<ber=1e-6, redundancy=3>
%b: !ft.i32<ber=1e-6, redundancy=3>
%sum: !ft.i32<ber=1e-6, redundancy=3> = arith.addi %a, %b : i32

// Lowering might produce:
// - Three separate add operations (one per copy)
// - A voting operation to combine results
```

#### Extension Challenge

Design a type that also captures:
- **Error correction code** (e.g., Hamming, Reed-Solomon)
- **Correction capability** (e.g., can correct 1 error, detect 2)

```mlir
!ft.i32<
  ber=1e-6,
  redundancy=3,
  ecc=hamming,
  correct_errors=1,
  detect_errors=2
>
```


# 🧱 PART 5 — Attributes & Metadata (Error Models Live Here)

### 5.1 Attributes vs Types

Understanding the difference between **attributes** and **types** is crucial for fault-tolerant IR design.

#### Types: Structural Meaning

**Types** describe the **intrinsic properties** of values:

- **What** the data represents (integer, float, qubit)
- **Structure** of the data (32-bit, array, tensor)
- **Inherent characteristics** (error rate, precision)

Types are **attached to values** and **flow through operations**:

```mlir
%a: !ft.i32<ber=1e-6>  // Type attached to value %a
%b: !ft.i32<ber=1e-6>  // Type attached to value %b
%sum: !ft.i32<ber=1e-6> = arith.addi %a, %b : i32  // Type flows through
```

**Key Property:** Types are **persistent** - they describe the value itself, not how it's used.

#### Attributes: Contextual Meaning

**Attributes** describe the **context of operations**:

- **How** an operation behaves (error model, optimization hints)
- **Configuration** for this specific operation
- **Metadata** that doesn't change the value's type

Attributes are **attached to operations** (not values):

```mlir
// Attribute on the operation, not the type
%result = faulty.add %a, %b {error_model = "bit-flip", ber = 1e-6} : i32
```

**Key Property:** Attributes are **local** - they describe this operation, not the result.

#### When to Use Types vs Attributes?

**Use Types When:**
- The fault property is **intrinsic to the value**
- The property **should propagate** through operations
- You want the **type system to enforce** correctness

**Example:**
```mlir
// Error rate is part of the value itself
%reliable: !ft.i32<ber=1e-9>
%unreliable: !ft.i32<ber=1e-3>
// Type system can prevent unsafe mixing
```

**Use Attributes When:**
- The fault property is **specific to this operation**
- The property is **operational metadata** (not value property)
- You want **flexibility** to vary per operation

**Example:**
```mlir
// Same values, different error models per operation
%a: i32
%b: i32
%sum1 = faulty.add %a, %b {error_model = "bit-flip"} : i32
%sum2 = faulty.add %a, %b {error_model = "stuck-at-0"} : i32
```

#### Hybrid Approach (Recommended)

Use **both** types and attributes:

- **Types** carry baseline fault properties
- **Attributes** override or extend for specific operations

```mlir
// Type provides baseline
%a: !ft.i32<ber=1e-6>
%b: !ft.i32<ber=1e-6>

// Attribute overrides for this specific operation
%sum = faulty.add %a, %b {override_ber = 1e-5} : !ft.i32<ber=1e-5>
```

### 5.2 Encoding Fault Models

Fault models describe **how hardware can fail**. Different fault models require different error correction strategies.

#### Common Fault Models

**1. Bit-Flip Errors**
- Random bit inversions (0→1 or 1→0)
- Common in: Radiation environments, memory errors
- Correction: Error-correcting codes (Hamming, Reed-Solomon)

```mlir
faulty.add %a, %b {error_model = "bit-flip", ber = 1e-6} : i32
```

**2. Stuck-At Faults**
- Bits stuck at 0 or 1
- Common in: Manufacturing defects, permanent failures
- Correction: Redundancy (TMR), diagnosis and replacement

```mlir
faulty.mul %a, %b {error_model = "stuck-at-0", prob = 0.001} : i32
```

**3. Transient Errors**
- Temporary faults that self-correct
- Common in: Soft errors, cosmic rays
- Correction: Error detection + retry, time-based mitigation

```mlir
faulty.load %mem[%idx] {error_model = "transient", rate = 0.0001} : memref<100xi32> -> i32
```

**4. Gate Errors (Quantum)**
- Imperfect gate operations
- Common in: Quantum computers
- Correction: Gate-level error correction, composite gates

```mlir
quantum.cx %q1, %q2 {gate_error = 0.001, error_model = "depolarizing"} : (!quantum.qubit, !quantum.qubit) -> ()
```

**5. Measurement Errors (Quantum)**
- Errors in reading quantum states
- Common in: Quantum measurement operations
- Correction: Repeated measurement, error-correction codes

```mlir
%result = quantum.measure %q {meas_error = 0.01} : !quantum.qubit -> i1
```

#### Encoding Fault Models in MLIR

**Option 1: String-Based Model Names**
```mlir
faulty.add %a, %b {error_model = "bit-flip"} : i32
```
- ✅ Simple, readable
- ❌ Not type-checked, no structure

**Option 2: Structured Attributes**
```mlir
faulty.add %a, %b {
  error_model = #fault.bit_flip<ber=1e-6>
} : i32
```
- ✅ Structured, can nest parameters
- ✅ Can be verified by type system
- ✅ More expressive

**Option 3: Dictionary Attributes**
```mlir
faulty.add %a, %b {
  error_model = {
    type = "bit-flip",
    ber = 1e-6,
    correction = "hamming"
  }
} : i32
```

#### Complete Example: Multiple Fault Models

```mlir
module {
  func.func @faulty_computation(%a: i32, %b: i32) -> i32 {
    // Bit-flip error model
    %sum = faulty.add %a, %b {
      error_model = "bit-flip",
      ber = 1e-6
    } : i32
    
    // Stuck-at fault model
    %prod = faulty.mul %sum, %b {
      error_model = "stuck-at-0",
      fault_prob = 0.001
    } : i32
    
    // Transient error model
    %mem = memref.alloc() : memref<100xi32>
    %loaded = faulty.load %mem[%sum] {
      error_model = "transient",
      error_rate = 0.0001
    } : memref<100xi32> -> i32
    
    func.return %loaded : i32
  }
}
```


In [3]:
# Examples of fault models encoded as attributes

print("=" * 70)
print("Fault Model Attributes in MLIR Operations")
print("=" * 70)

# Example 1: Bit-flip error model
bit_flip_example = """
faulty.add %a, %b {
  error_model = "bit-flip",
  ber = 1e-6
} : i32
"""
print("\n1. Bit-Flip Error Model:")
print(bit_flip_example)

# Example 2: Stuck-at fault model
stuck_at_example = """
faulty.mul %a, %b {
  error_model = "stuck-at-0",
  fault_prob = 0.001
} : i32
"""
print("\n2. Stuck-At Fault Model:")
print(stuck_at_example)

# Example 3: Transient error model
transient_example = """
faulty.load %mem[%idx] {
  error_model = "transient",
  error_rate = 0.0001
} : memref<100xi32> -> i32
"""
print("\n3. Transient Error Model:")
print(transient_example)

# Example 4: Quantum gate error
quantum_example = """
quantum.cx %q1, %q2 {
  error_model = "depolarizing",
  gate_error = 0.001
} : (!quantum.qubit, !quantum.qubit) -> ()
"""
print("\n4. Quantum Gate Error Model:")
print(quantum_example)

# Example 5: Structured attribute (more advanced)
structured_example = """
faulty.add %a, %b {
  error_model = #fault.bit_flip<ber=1e-6, correction=hamming>
} : i32
"""
print("\n5. Structured Attribute (Type-Safe):")
print(structured_example)

print("\n" + "=" * 70)


Fault Model Attributes in MLIR Operations

1. Bit-Flip Error Model:

faulty.add %a, %b {
  error_model = "bit-flip",
  ber = 1e-6
} : i32


2. Stuck-At Fault Model:

faulty.mul %a, %b {
  error_model = "stuck-at-0",
  fault_prob = 0.001
} : i32


3. Transient Error Model:

faulty.load %mem[%idx] {
  error_model = "transient",
  error_rate = 0.0001
} : memref<100xi32> -> i32


4. Quantum Gate Error Model:

quantum.cx %q1, %q2 {
  error_model = "depolarizing",
  gate_error = 0.001
} : (!quantum.qubit, !quantum.qubit) -> ()


5. Structured Attribute (Type-Safe):

faulty.add %a, %b {
  error_model = #fault.bit_flip<ber=1e-6, correction=hamming>
} : i32




### 5.3 Static vs Dynamic Fault Metadata

Fault metadata can be **static** (known at compile time) or **dynamic** (determined at runtime). Each approach has trade-offs.

#### Static Fault Metadata (Compile-Time)

**Characteristics:**
- Known when compiling
- Fixed values or constants
- Can be verified and optimized at compile time

**Use Cases:**
- Hardware specifications (gate fidelities from data sheets)
- Design-time error budgets
- System reliability requirements

**Examples:**

```mlir
// Static BER from hardware specs
%result = faulty.add %a, %b {ber = 1e-6} : i32

// Static error model
quantum.h %q {gate_error = 0.001} : !quantum.qubit -> !quantum.qubit

// Static reliability requirement
%computed = ft.compute %data {
  min_reliability = 0.999,
  max_ber = 1e-9
} : !ft.tensor<100xi32> -> !ft.tensor<100xi32>
```

**Advantages:**
- ✅ Compile-time verification
- ✅ Optimize for known constraints
- ✅ Catch errors early
- ✅ Better code generation

**Limitations:**
- ❌ Can't adapt to runtime conditions
- ❌ Assumes stable error rates
- ❌ May be pessimistic or optimistic

#### Dynamic Fault Metadata (Runtime)

**Characteristics:**
- Determined during execution
- Values come from runtime measurements
- Requires runtime infrastructure

**Use Cases:**
- Adaptive error correction
- Runtime error monitoring
- Performance/reliability trade-offs
- Self-healing systems

**Examples:**

```mlir
// Runtime error measurement
%measured_ber = faulty.measure_error_rate %op : !faulty.op -> f64

// Dynamic error model selection
%result = scf.if %measured_ber > %threshold -> i32 {
  // High error: use strong correction
  faulty.add_strong %a, %b {correction = "tmr"} : i32
} else {
  // Low error: use weak correction
  faulty.add_weak %a, %b {correction = "parity"} : i32
}

// Runtime reliability adjustment
%adaptive_result = faulty.adaptive_add %a, %b {
  runtime_monitoring = true
} : i32 -> i32
```

**Advantages:**
- ✅ Adapts to actual conditions
- ✅ Can optimize dynamically
- ✅ Handles varying environments

**Limitations:**
- ❌ Requires runtime infrastructure
- ❌ Harder to verify statically
- ❌ Overhead for monitoring
- ❌ More complex implementation

#### Hybrid Approach (Recommended)

Combine static and dynamic metadata:

```mlir
module {
  func.func @adaptive_fault_tolerant(%a: i32, %b: i32) -> i32 {
    // Static: Baseline error model from specs
    %baseline_ber = arith.constant 1e-6 : f64
    
    // Dynamic: Measure actual error rate
    %measured_ber = faulty.measure_error_rate %a : i32 -> f64
    
    // Compare and choose strategy
    %needs_strong_correction = arith.cmpf ogt, %measured_ber, %baseline_ber : f64
    
    %result = scf.if %needs_strong_correction -> i32 {
      // Use strong correction if errors are worse than expected
      faulty.add %a, %b {
        error_model = "bit-flip",
        ber = %measured_ber,  // Dynamic BER
        correction = "tmr"     // Static correction strategy
      } : i32
    } else {
      // Use standard correction
      faulty.add %a, %b {
        error_model = "bit-flip",
        ber = %baseline_ber,   // Static BER
        correction = "parity"  // Static correction strategy
      } : i32
    }
    
    func.return %result : i32
  }
}
```

#### Compile-Time Guarantees

Even with dynamic metadata, you can provide **compile-time guarantees**:

**1. Worst-Case Analysis**
```mlir
// Compiler can verify: "Even with worst-case errors, system meets requirements"
%result = faulty.compute %data {
  max_ber = 1e-3,        // Worst-case assumption
  min_reliability = 0.99 // Guaranteed minimum
} : !ft.tensor<100xi32> -> !ft.tensor<100xi32>
```

**2. Type-Level Guarantees**
```mlir
// Type system enforces: "Result always has BER <= 1e-6"
%a: !ft.i32<ber=1e-6>
%b: !ft.i32<ber=1e-6>
%result: !ft.i32<ber<=1e-6> = faulty.add %a, %b : i32
```

**3. Verification Passes**
```mlir
// Verification pass checks: "Does this meet reliability budget?"
ft.verify_reliability @function {
  required_reliability = 0.999,
  error_budget = 1e-6
}
```

#### Runtime Instrumentation Hooks

For dynamic metadata, you need runtime hooks:

**1. Error Measurement**
```mlir
// Runtime hook: Measure actual error rate
%actual_ber = faulty.measure_ber %value : !ft.i32<ber=1e-6> -> f64
```

**2. Error Injection (Testing)**
```mlir
// Runtime hook: Inject errors for testing
%corrupted = faulty.inject_error %value {
  error_rate = 0.1,  // High rate for testing
  error_model = "bit-flip"
} : i32 -> i32
```

**3. Monitoring Callbacks**
```mlir
// Runtime hook: Callback when errors detected
faulty.add %a, %b {
  error_model = "bit-flip",
  on_error = @error_handler  // Runtime callback
} : i32
```

**4. Adaptive Correction**
```mlir
// Runtime hook: Adjust correction strength
%result = faulty.adaptive_add %a, %b {
  monitor = true,
  adjust_correction = true,
  min_reliability = 0.99
} : i32 -> i32
```

#### Design Recommendations

**For Your Fault-Tolerant Dialect:**

1. **Start with Static** - Easier to implement and verify
2. **Add Dynamic Later** - For adaptive systems
3. **Use Hybrid** - Static baseline + dynamic adjustment
4. **Provide Hooks** - Runtime measurement and monitoring
5. **Maintain Guarantees** - Even with dynamic metadata, provide compile-time bounds

**Example Dialect Design:**
```mlir
// Static baseline
faulty.add %a, %b {ber = 1e-6} : i32

// Dynamic override
faulty.add %a, %b {ber = %runtime_ber} : i32

// Hybrid with guarantees
faulty.add %a, %b {
  static_ber = 1e-6,      // Compile-time baseline
  runtime_ber = %measured, // Runtime measurement
  max_ber = 1e-4,         // Compile-time guarantee (worst case)
  min_reliability = 0.99  // Compile-time guarantee
} : i32
```


# 🧱 PART 6 — Verification: Enforcing Fault Rules

### 6.1 Why Verification Is Critical

Fault tolerance cannot be achieved through **hope** or **best practices** — it must be **provably enforced** by the compiler infrastructure.

#### The Problem with "Hope-Based" Fault Tolerance

**Traditional Approach (Broken):**
```c
// Programmer hopes the hardware is reliable
int result = a + b;  // What if this operation has a bit-flip?
```

**Problems:**
- No way to verify fault tolerance properties
- Errors discovered only at runtime (too late!)
- Can't optimize for reliability
- Can't enforce constraints

#### Verification-Enforced Fault Tolerance

**MLIR Approach:**
```mlir
// Compiler verifies fault tolerance before code generation
%result = faulty.add %a, %b {ber = 1e-6} : !ft.i32<ber=1e-6>
// Verification pass ensures: error correction is present, reliability meets requirements
```

**Benefits:**
- ✅ Catch errors at compile time
- ✅ Prove fault tolerance properties
- ✅ Enforce correctness constraints
- ✅ Enable reliability-aware optimization

#### What Verification Must Check

**1. Type Safety**
- Are fault-tolerant types used correctly?
- Do operations preserve fault properties?
- Are incompatible types prevented from mixing?

**2. Error Budget Compliance**
- Does the computation stay within error budget?
- Are reliability requirements met?
- Is error accumulation bounded?

**3. Lowering Safety**
- Can this operation be safely lowered to hardware?
- Is error correction present where needed?
- Are logical operations protected before lowering to physical?

**4. Consistency**
- Do types match their declared properties?
- Are attributes consistent with operations?
- Do nested operations maintain invariants?

### 6.2 Operation Verification Hooks

MLIR provides **verification hooks** where you can enforce fault tolerance rules. These hooks run automatically when operations are created or modified.

#### Basic Verification Pattern

In MLIR, each operation can define a `verify()` method that checks correctness:

```cpp
// Conceptual C++ code (what happens under the hood)
LogicalResult FaultyAddOp::verify() {
  // Check: operands must have compatible fault types
  auto lhsType = getLhs().getType().cast<FaultTolerantType>();
  auto rhsType = getRhs().getType().cast<FaultTolerantType>();
  
  if (lhsType.getBER() != rhsType.getBER()) {
    return emitError("BER mismatch: cannot add values with different error rates");
  }
  
  // Check: operation must have error correction if BER exceeds threshold
  if (lhsType.getBER() > 1e-6) {
    if (!hasAttr("correction")) {
      return emitError("Operations with BER > 1e-6 require error correction");
    }
  }
  
  return success();
}
```

#### Example 1: Disallow Unsafe Lowering

**Rule:** Logical operations cannot be lowered to physical hardware without error correction.

```mlir
// This should fail verification
module {
  func.func @unsafe_lowering() {
    // Logical qubit operation
    %q: !quantum.qubit<level=logical> = quantum.alloc_qubit()
    
    // Attempt to lower directly to physical (UNSAFE!)
    // Verification should catch this
    quantum.lower_to_physical %q : !quantum.qubit<level=physical>
    // ERROR: Logical operations require error correction before physical lowering
  }
}
```

**Verification Logic:**
```mlir
// Verification rule (conceptual)
verify_quantum_lowering(op) {
  if (op.input_type == logical && op.target_type == physical) {
    if (!has_error_correction(op)) {
      emit_error("Logical to physical lowering requires error correction");
      return failure();
    }
  }
  return success();
}
```

#### Example 2: Enforce Redundancy Constraints

**Rule:** Operations on values with `redundancy=3` must use 3-way voting.

```mlir
module {
  func.func @redundancy_check() {
    %a: !ft.i32<ber=1e-6, redundancy=3>
    %b: !ft.i32<ber=1e-6, redundancy=3>
    
    // This should fail verification
    %result = faulty.add %a, %b {correction="parity"} : i32
    // ERROR: Values with redundancy=3 require TMR (triple modular redundancy)
    
    // This should pass verification
    %correct = faulty.add %a, %b {correction="tmr", voting="majority"} : i32
  }
}
```

**Verification Logic:**
```mlir
verify_redundancy_constraint(op) {
  redundancy = get_max_redundancy(op.operands);
  if (redundancy > 1) {
    if (op.correction != "tmr" && redundancy == 3) {
      emit_error("redundancy=3 requires TMR correction");
      return failure();
    }
  }
  return success();
}
```

#### Example 3: Error Budget Enforcement

**Rule:** Total error accumulation must not exceed the reliability budget.

```mlir
module {
  func.func @error_budget_check() {
    // Initial values with BER
    %a: !ft.i32<ber=1e-6>
    %b: !ft.i32<ber=1e-6>
    
    // Each operation accumulates error
    %sum = faulty.add %a, %b : !ft.i32<ber=2e-6>      // Error accumulates
    %prod = faulty.mul %sum, %b : !ft.i32<ber=3e-6>   // More accumulation
    %result = faulty.div %prod, %a : !ft.i32<ber=4e-6> // Even more
    
    // Verification should check: does 4e-6 exceed the budget?
    // If budget is 1e-5, this passes
    // If budget is 1e-6, this fails
  }
}
```

**Verification Logic:**
```mlir
verify_error_budget(function) {
  error_budget = function.getAttr("max_ber");
  accumulated_error = compute_accumulated_error(function);
  
  if (accumulated_error > error_budget) {
    emit_error("Error budget exceeded: accumulated " + 
               accumulated_error + " > budget " + error_budget);
    return failure();
  }
  return success();
}
```

#### Example 4: Type Compatibility Rules

**Rule:** Operations must handle type compatibility correctly.

```mlir
module {
  func.func @type_compatibility() {
    %reliable: !ft.i32<ber=1e-9>      // Very reliable
    %unreliable: !ft.i32<ber=1e-3>    // Not reliable
    
    // Option 1: Explicit conversion (should pass)
    %converted = faulty.upgrade_reliability %unreliable : !ft.i32<ber=1e-9>
    %sum1 = faulty.add %reliable, %converted : i32
    
    // Option 2: Direct operation (should fail verification)
    %sum2 = faulty.add %reliable, %unreliable : i32
    // ERROR: Cannot mix values with incompatible BER without explicit conversion
  }
}
```

#### Complete Verification Example

Here's a complete example showing multiple verification rules:

```mlir
module {
  // Error budget for this module
  attributes {max_ber = 1e-5}
  
  func.func @verified_computation(%a: !ft.i32<ber=1e-6>, 
                                   %b: !ft.i32<ber=1e-6>) -> !ft.i32 {
    // Verification checks:
    // 1. Input types compatible? (both have ber=1e-6) ✓
    // 2. Operation has correction? (needed for ber=1e-6) ✓
    // 3. Result type matches? ✓
    %sum = faulty.add %a, %b {
      error_model = "bit-flip",
      correction = "parity"
    } : !ft.i32<ber=2e-6>
    
    // Verification checks:
    // 4. Error accumulation: 2e-6 < 1e-5? ✓
    // 5. Can this be lowered? (has correction) ✓
    
    func.return %sum : !ft.i32
  }
}
```

#### Implementing Verification in Your Dialect

When designing your `faulty` dialect, implement verification for:

1. **Type Compatibility** - Operands have compatible fault properties
2. **Error Correction Requirements** - High-error operations have correction
3. **Error Budget** - Accumulated error stays within budget
4. **Lowering Safety** - Operations can be safely lowered
5. **Redundancy Consistency** - Redundancy requirements are met
6. **Attribute Consistency** - Attributes match operation semantics

**Key Principle:**

> **Verification is not optional** — it's the mechanism that makes fault tolerance provable, not hopeful.


# 🧱 PART 7 — Passes: Analysis & Transformation

### 7.1 MLIR Pass Infrastructure (Conceptual)

MLIR uses a **pass-based architecture** for analyzing and transforming IR. Passes are the mechanism that makes fault tolerance happen.

#### Two Types of Passes

**1. Analysis Passes**
- **Purpose**: Understand the program (don't modify it)
- **Output**: Information about the program (error rates, dependencies, etc.)
- **Example**: "What is the error accumulation in this function?"

**2. Transformation Passes**
- **Purpose**: Modify the program (improve it)
- **Output**: Modified IR with optimizations or corrections
- **Example**: "Insert error correction where needed"

#### Pass Execution Model

```
Input IR
    ↓
[Analysis Pass] → Collect information
    ↓
[Transformation Pass] → Use information to transform
    ↓
[Analysis Pass] → Verify transformation
    ↓
[Transformation Pass] → Further optimizations
    ↓
Output IR
```

**Key Property:** Passes can be **composed** into pipelines. You run multiple passes in sequence to achieve complex transformations.

### 7.2 Fault-Aware Analysis Passes

Analysis passes gather information about fault tolerance properties without modifying the code.

#### 1. Error Accumulation Analysis

**Goal:** Track how errors propagate and accumulate through operations.

**What it analyzes:**
- Starting error rates (from types/attributes)
- Error accumulation at each operation
- Total error at function outputs
- Critical paths (paths with highest error accumulation)

**Example:**

```mlir
module {
  func.func @analyze_error_accumulation(%a: !ft.i32<ber=1e-6>, 
                                        %b: !ft.i32<ber=1e-6>) -> !ft.i32 {
    // Analysis tracks:
    // Input: ber = 1e-6 (both operands)
    
    %sum = faulty.add %a, %b : !ft.i32<ber=2e-6>
    // Analysis: add operation doubles error (no correction)
    // Accumulated: 2e-6
    
    %prod = faulty.mul %sum, %b : !ft.i32<ber=3e-6>
    // Analysis: mul operation adds 1e-6 more error
    // Accumulated: 3e-6
    
    func.return %prod : !ft.i32
    // Analysis output: Function accumulates 3e-6 total error
  }
}
```

**Analysis Output:**
```
Error Accumulation Report:
  Function: @analyze_error_accumulation
  Input BER: 1e-6
  Output BER: 3e-6
  Error Multiplication Factor: 3.0x
  Critical Path: %a → add → mul → return
```

#### 2. Reliability Budget Analysis

**Goal:** Check if the computation stays within reliability constraints.

**What it analyzes:**
- Error budget (maximum allowed error)
- Current error usage
- Budget violations
- Recommendations for correction

**Example:**

```mlir
module {
  attributes {reliability_budget = 1e-5}  // Maximum 1e-5 error allowed
  
  func.func @budget_check(%a: !ft.i32<ber=1e-6>) -> !ft.i32 {
    %result = faulty.compute_chain %a : !ft.i32<ber=8e-6>
    // Analysis: 8e-6 > 1e-5 budget → VIOLATION
    func.return %result : !ft.i32
  }
}
```

**Analysis Output:**
```
Reliability Budget Analysis:
  Function: @budget_check
  Budget: 1e-5
  Used: 8e-6
  Remaining: 2e-6
  Status: WITHIN BUDGET ✓
  
  Warning: Approaching budget limit
  Recommendation: Consider error correction
```

#### 3. Critical Path Vulnerability Analysis

**Goal:** Identify the most vulnerable paths (paths most likely to fail).

**What it analyzes:**
- Paths through the computation
- Error rates on each path
- Vulnerability ranking
- Suggestions for hardening

**Example:**

```mlir
module {
  func.func @critical_path(%a: !ft.i32<ber=1e-6>, 
                           %b: !ft.i32<ber=1e-6>) -> (!ft.i32, !ft.i32) {
    // Path 1: Low error path
    %sum = faulty.add %a, %b {correction="parity"} : !ft.i32<ber=1e-7>
    
    // Path 2: High error path (no correction)
    %prod = faulty.mul %a, %b : !ft.i32<ber=2e-6>
    
    func.return %sum, %prod : !ft.i32, !ft.i32
  }
}
```

**Analysis Output:**
```
Critical Path Vulnerability:
  Path 1: %a → add → return (ber=1e-7) → LOW RISK ✓
  Path 2: %a → mul → return (ber=2e-6) → HIGH RISK ⚠
  
  Recommendation: Add error correction to multiplication
```

#### 4. Error Model Consistency Analysis

**Goal:** Check that error models are used consistently.

**What it analyzes:**
- Are error models consistent across operations?
- Do correction strategies match error models?
- Are there conflicting assumptions?

**Example Analysis:**

```
Error Model Consistency:
  Operation: faulty.add
    Error Model: "bit-flip"
    Correction: "parity"
    Status: COMPATIBLE ✓
  
  Operation: faulty.mul
    Error Model: "stuck-at-0"
    Correction: "parity"
    Status: INCOMPATIBLE ⚠
    Issue: Parity codes don't detect stuck-at faults
    Recommendation: Use TMR for stuck-at faults
```

### 7.3 Transformation Passes

Transformation passes modify the IR to improve fault tolerance.

#### 1. Insert Redundancy Pass

**Goal:** Automatically add redundancy where needed.

**Transformation:**

```mlir
// Before: No redundancy
func.func @add_values(%a: i32, %b: i32) -> i32 {
  %result = arith.addi %a, %b : i32
  return %result : i32
}

// After: Triple Modular Redundancy inserted
func.func @add_values(%a: i32, %b: i32) -> i32 {
  // Three copies of the operation
  %result1 = arith.addi %a, %b : i32
  %result2 = arith.addi %a, %b : i32
  %result3 = arith.addi %a, %b : i32
  
  // Voting to select correct result
  %result = faulty.majority_vote %result1, %result2, %result3 : i32
  return %result : i32
}
```

#### 2. Add Error Correction Pass

**Goal:** Insert error correction operations where reliability requirements demand it.

**Transformation:**

```mlir
// Before: High error, no correction
%result = faulty.add %a, %b {ber=1e-4} : !ft.i32<ber=1e-4>

// After: Error correction inserted
%raw_result = faulty.add %a, %b {ber=1e-4} : !ft.i32<ber=1e-4>
%encoded = faulty.hamming_encode %raw_result : !ft.i32<ber=1e-4> -> !ft.encoded<i32>
%decoded = faulty.hamming_decode %encoded : !ft.encoded<i32> -> !ft.i32<ber=1e-6>
%result = %decoded
```

#### 3. Replace Unsafe Operations Pass

**Goal:** Replace operations that violate fault tolerance constraints with safe alternatives.

**Transformation:**

```mlir
// Before: Unsafe operation (no error handling)
%result = faulty.div %a, %b {ber=1e-3} : !ft.i32<ber=1e-3>

// After: Safe operation with error checking
%checked_b = faulty.check_nonzero %b : i32 -> i1
%result = scf.if %checked_b -> i32 {
  %div_result = faulty.div %a, %b {ber=1e-3, error_handling="detect"} : i32
  scf.yield %div_result : i32
} else {
  %error_val = arith.constant 0 : i32
  faulty.report_error "Division by zero detected"
  scf.yield %error_val : i32
}
```

#### 4. Error Budget Optimization Pass

**Goal:** Optimize error correction to stay within budget while minimizing overhead.

**Transformation:**

```mlir
// Before: Over-correction (unnecessary overhead)
%result1 = faulty.add %a, %b {correction="tmr"} : i32  // TMR for low-error op
%result2 = faulty.mul %result1, %c {correction="tmr"} : i32

// After: Optimized correction (right amount for each operation)
%result1 = faulty.add %a, %b {correction="parity"} : i32  // Parity sufficient
%result2 = faulty.mul %result1, %c {correction="tmr"} : i32  // TMR for high-error op
```

#### Complete Pass Pipeline Example

A typical fault-tolerant compilation pipeline:

```
1. Error Accumulation Analysis
   → Understand current error levels

2. Insert Redundancy Pass
   → Add redundancy to critical operations

3. Add Error Correction Pass
   → Insert correction where needed

4. Reliability Budget Analysis
   → Verify we're within budget

5. Error Budget Optimization Pass
   → Optimize correction levels

6. Critical Path Analysis
   → Identify remaining vulnerabilities

7. Final Verification Pass
   → Ensure all constraints met
```

📌 **Key Idea**

> *Fault tolerance is implemented as passes, not magic.*

Fault tolerance in MLIR is achieved by:
1. **Analysis passes** understand fault properties
2. **Transformation passes** modify code to improve reliability
3. **Pass pipelines** compose multiple passes for complete solutions
4. **Iterative refinement** through multiple analysis/transformation cycles

This is the same pattern used for traditional compiler optimizations — just applied to fault tolerance instead of performance.


# 🧱 PART 8 — Progressive Lowering (The MLIR Superpower)

### 8.1 What Lowering Really Means

**Lowering** is the process of transforming IR from high-level, abstract operations to low-level, hardware-specific operations. In MLIR, lowering is **progressive** — you can have many levels of abstraction.

#### The Lowering Hierarchy

```
High-level semantics (Algorithmic intent)
    ↓ [Lowering Pass 1]
Domain-specific semantics (Fault-aware operations)
    ↓ [Lowering Pass 2]
Logical semantics (Error-corrected operations)
    ↓ [Lowering Pass 3]
Physical semantics (Hardware-mapped operations)
    ↓ [Lowering Pass 4]
Executable constraints (Machine code)
```

#### Why Progressive Lowering Matters

**Traditional Compilers (Single Step):**
```
Source Code → Machine Code
    (One big jump, loses information)
```

**MLIR (Progressive Steps):**
```
Source Code → Dialect 1 → Dialect 2 → Dialect 3 → Machine Code
    (Each step preserves and transforms information)
```

**Benefits:**
- ✅ Can reason at each level
- ✅ Can optimize at each level
- ✅ Can verify at each level
- ✅ Information isn't lost prematurely

#### Example: What Changes During Lowering

**Level 1: High-Level Intent**
```mlir
// What: "Compute matrix multiplication"
%result = linalg.matmul %A, %B : tensor<100x200xf32>, tensor<200x300xf32>
```

**Level 2: Fault-Aware**
```mlir
// What: "Compute with error protection"
%result = ft.matmul %A, %B {
  error_budget = 1e-6,
  correction = "hamming"
} : !ft.tensor<100x200xf32>, !ft.tensor<200x300xf32> -> !ft.tensor<100x300xf32>
```

**Level 3: Error-Corrected Operations**
```mlir
// What: "Explicit error correction steps"
%encoded_A = ft.encode %A {code = "hamming"} : tensor<100x200xf32>
%encoded_B = ft.encode %B {code = "hamming"} : tensor<200x300xf32>
%raw_result = linalg.matmul %encoded_A, %encoded_B : tensor<...>
%result = ft.decode %raw_result {code = "hamming"} : tensor<100x300xf32>
```

**Level 4: Hardware-Specific**
```mlir
// What: "Hardware operations with timing"
%result = hardware.matmul %A, %B {
  accelerator = "tpu",
  error_rate = 1e-6,
  timing_constraints = {...}
} : memref<...>, memref<...> -> memref<...>
```

**Level 5: Executable (LLVM)**
```llvm
; Machine code with error checking
call @matmul_kernel(%A, %B, %result)
call @check_errors(%result)
```

### 8.2 Example Lowering Pipeline

Here's a complete example of a fault-tolerant lowering pipeline:

#### Pipeline Overview

```
ft-dialect (Fault-Tolerant)
    ↓ [ft-to-logical pass]
logical-dialect (Error-Corrected Logical Operations)
    ↓ [logical-to-physical pass]
physical-dialect (Hardware-Mapped Physical Operations)
    ↓ [physical-to-llvm pass]
LLVM Dialect (Executable Code)
    ↓ [llvm-to-machine pass]
Machine Code
```

#### Step-by-Step Transformation

**Step 1: Fault-Tolerant Dialect (Starting Point)**

```mlir
module {
  func.func @compute(%a: !ft.i32<ber=1e-6>, %b: !ft.i32<ber=1e-6>) -> !ft.i32 {
    %result = ft.add %a, %b {
      error_budget = 1e-5,
      min_reliability = 0.99
    } : !ft.i32<ber=1e-6> -> !ft.i32<ber=2e-6>
    
    func.return %result : !ft.i32
  }
}
```

**Step 2: Lower to Logical Dialect (After ft-to-logical pass)**

```mlir
module {
  func.func @compute(%a: !logical.i32, %b: !logical.i32) -> !logical.i32 {
    // Error correction inserted
    %a_encoded = logical.encode %a {code = "parity"} : !logical.i32 -> !logical.encoded<i32>
    %b_encoded = logical.encode %b {code = "parity"} : !logical.i32 -> !logical.encoded<i32>
    
    // Logical addition operation
    %encoded_result = logical.add %a_encoded, %b_encoded : !logical.encoded<i32>
    
    // Error detection and correction
    %checked = logical.check_and_correct %encoded_result {code = "parity"} : !logical.encoded<i32>
    %result = logical.decode %checked {code = "parity"} : !logical.encoded<i32> -> !logical.i32
    
    func.return %result : !logical.i32
  }
}
```

**Step 3: Lower to Physical Dialect (After logical-to-physical pass)**

```mlir
module {
  func.func @compute(%a: i32, %b: i32) -> i32 {
    // Physical encoding (hardware-specific)
    %a_parity = physical.compute_parity %a : i32 -> i8
    %b_parity = physical.compute_parity %b : i32 -> i8
    
    // Physical addition
    %sum = arith.addi %a, %b : i32
    %sum_parity = physical.compute_parity %sum : i32 -> i8
    
    // Physical error checking
    %expected_parity = physical.xor_parity %a_parity, %b_parity : i8
    %parity_match = arith.cmpi eq, %sum_parity, %expected_parity : i8
    %result = scf.if %parity_match -> i32 {
      scf.yield %sum : i32
    } else {
      // Error detected: use correction or signal error
      %corrected = physical.correct_error %sum, %a_parity, %b_parity : i32
      scf.yield %corrected : i32
    }
    
    func.return %result : i32
  }
}
```

**Step 4: Lower to LLVM Dialect (After physical-to-llvm pass)**

```mlir
module {
  llvm.func @compute(%a: !llvm.i32, %b: !llvm.i32) -> !llvm.i32 {
    // LLVM operations (close to machine code)
    %a_parity = llvm.call @compute_parity(%a) : (!llvm.i32) -> !llvm.i8
    %b_parity = llvm.call @compute_parity(%b) : (!llvm.i32) -> !llvm.i8
    %sum = llvm.add %a, %b : !llvm.i32
    %sum_parity = llvm.call @compute_parity(%sum) : (!llvm.i32) -> !llvm.i8
    %expected = llvm.xor %a_parity, %b_parity : !llvm.i8
    %match = llvm.icmp "eq" %sum_parity, %expected : !llvm.i1
    %result = llvm.cond_br %match, ^bb1, ^bb2
    ^bb1:
      llvm.return %sum : !llvm.i32
    ^bb2:
      %corrected = llvm.call @correct_error(%sum, %a_parity, %b_parity) : (!llvm.i32, !llvm.i8, !llvm.i8) -> !llvm.i32
      llvm.return %corrected : !llvm.i32
  }
}
```

#### What Changed at Each Level

| Level | Types | Operations | Error Handling | Hardware Awareness |
|-------|-------|------------|----------------|-------------------|
| **ft-dialect** | `!ft.i32<ber=X>` | `ft.add` | Declarative (attributes) | None |
| **logical-dialect** | `!logical.i32`, `!logical.encoded<i32>` | `logical.encode`, `logical.add`, `logical.check_and_correct` | Explicit correction ops | None |
| **physical-dialect** | `i32`, `i8` (parity) | `physical.compute_parity`, `physical.xor_parity`, `physical.correct_error` | Conditional branches | Partial (knows about parity computation) |
| **LLVM** | `!llvm.i32`, `!llvm.i8`, `!llvm.i1` | `llvm.call`, `llvm.add`, `llvm.cond_br` | Function calls + branches | Full (ready for codegen) |

### 8.3 Fault-Aware Lowering Decisions

During lowering, the compiler must make decisions about **how to handle faults**. These decisions depend on error rates, reliability requirements, and hardware capabilities.

#### Decision Point 1: When to Correct

**Question:** Should we insert error correction at this lowering step?

**Criteria:**
- Error rate exceeds threshold?
- Reliability requirement not met?
- Hardware supports correction?

**Example Decision:**

```mlir
// High error rate → Insert correction
%a: !ft.i32<ber=1e-3>  // High error
%result = ft.add %a, %b
// Decision: INSERT CORRECTION
// Lowering produces: encode → add → check → correct → decode

// Low error rate → Skip correction (save resources)
%b: !ft.i32<ber=1e-9>  // Very low error
%result = ft.add %a, %b
// Decision: SKIP CORRECTION (optional)
// Lowering produces: add (no correction overhead)
```

#### Decision Point 2: When to Mitigate

**Question:** Can we reduce error impact without full correction?

**Strategies:**
- Use weaker correction (parity instead of TMR)
- Accept higher error rate in non-critical paths
- Use approximate operations where precision isn't critical

**Example Decision:**

```mlir
// Critical path → Full correction
%critical = ft.compute_critical %data {
  min_reliability = 0.999
} : !ft.tensor<100xf32>
// Decision: FULL CORRECTION (TMR)

// Non-critical path → Mitigation only
%auxiliary = ft.compute_auxiliary %data {
  min_reliability = 0.95  // Lower requirement
} : !ft.tensor<100xf32>
// Decision: PARTIAL CORRECTION (parity check)
```

#### Decision Point 3: When to Fail Fast

**Question:** Should we detect errors early and abort, or try to continue?

**Strategies:**
- Fail fast: Detect error → abort immediately (for safety-critical)
- Continue: Detect error → attempt correction → continue (for availability)
- Degrade: Detect error → use degraded mode (for graceful degradation)

**Example Decision:**

```mlir
// Safety-critical → Fail fast
%safe_result = ft.safe_compute %data {
  on_error = "abort",  // Fail immediately on error
  error_handler = @abort_handler
} : !ft.tensor<100xf32>

// High-availability → Continue with correction
%available_result = ft.available_compute %data {
  on_error = "correct_and_continue",  // Try to fix and continue
  max_correction_attempts = 3
} : !ft.tensor<100xf32>

// Best-effort → Degrade gracefully
%degraded_result = ft.best_effort_compute %data {
  on_error = "degrade",  // Use simpler computation
  fallback_mode = "approximate"
} : !ft.tensor<100xf32>
```

#### Decision Matrix

| Error Rate | Reliability Requirement | Hardware Support | Decision |
|------------|------------------------|------------------|----------|
| High (1e-3) | High (0.999) | Full correction | **Full correction** |
| High (1e-3) | Medium (0.99) | Full correction | **Mitigation** (weaker correction) |
| Medium (1e-6) | High (0.999) | Partial correction | **Mitigation** (available correction) |
| Medium (1e-6) | Medium (0.99) | Partial correction | **Skip correction** (optional) |
| Low (1e-9) | Any | Any | **Skip correction** (unnecessary) |
| Any | Safety-critical | Any | **Fail fast** (on error) |
| Any | High-availability | Any | **Continue** (with correction) |

#### Lowering Pass Implementation Pattern

Each lowering pass follows this pattern:

```
1. Analyze: What are the fault tolerance requirements?
2. Decide: What strategy should we use? (correct/mitigate/fail-fast)
3. Transform: Insert appropriate operations
4. Verify: Do transformed operations meet requirements?
```

**Example Lowering Pass Logic (Pseudocode):**

```
lower_ft_to_logical(operation) {
  // 1. Analyze
  error_rate = get_error_rate(operation);
  reliability_req = get_reliability_requirement(operation);
  
  // 2. Decide
  if (error_rate > threshold || reliability_req > 0.99) {
    strategy = FULL_CORRECTION;
  } else if (reliability_req > 0.95) {
    strategy = PARTIAL_CORRECTION;
  } else {
    strategy = NO_CORRECTION;
  }
  
  // 3. Transform
  if (strategy == FULL_CORRECTION) {
    insert_encode(operation);
    insert_check_and_correct(operation);
    insert_decode(operation);
  } else if (strategy == PARTIAL_CORRECTION) {
    insert_parity_check(operation);
  }
  
  // 4. Verify
  verify_reliability_met(transformed_operation);
}
```

**Key Principle:**

> **Lowering is not just translation** — it's making **strategic decisions** about fault tolerance based on requirements, constraints, and hardware capabilities.


# 🧱 PART 9 — Case Study: Fault-Tolerant Compute Pipeline

### 9.1 Problem Statement

Let's build a **complete fault-tolerant arithmetic kernel** that computes `result = (a + b) * c` with fault tolerance guarantees.

#### Requirements

**Functional Requirements:**
- Compute: `result = (a + b) * c`
- Handle 32-bit integer inputs
- Return 32-bit integer result

**Fault Tolerance Requirements:**
- Input error rate: `ber = 1e-6` (bit-error rate 10⁻⁶)
- Maximum output error rate: `ber <= 1e-5`
- Minimum reliability: `0.99` (99% confidence)
- Error model: Bit-flip errors
- Correction strategy: Parity checking (detection) + retry (correction)

**Constraints:**
- Limited reliability budget: `1e-5` total error
- Must stay within budget
- Prefer lightweight correction (parity) over heavy correction (TMR)

### 9.2 IR Evolution Across Levels

Let's trace how the IR evolves through the compilation pipeline, showing the complete transformation from high-level intent to executable code.

#### Level 1: High-Level Intent (ft-dialect)

**Starting Point:** Algorithm with fault tolerance requirements

```mlir
module {
  attributes {
    reliability_budget = 1e-5,
    min_reliability = 0.99
  }
  
  func.func @compute(%a: !ft.i32<ber=1e-6>, 
                     %b: !ft.i32<ber=1e-6>, 
                     %c: !ft.i32<ber=1e-6>) -> !ft.i32 {
    // Step 1: Addition
    %sum = ft.add %a, %b {
      error_model = "bit-flip",
      min_reliability = 0.99
    } : !ft.i32<ber=1e-6> -> !ft.i32<ber=2e-6>
    
    // Step 2: Multiplication
    %result = ft.mul %sum, %c {
      error_model = "bit-flip",
      min_reliability = 0.99
    } : !ft.i32<ber=2e-6> -> !ft.i32<ber=3e-6>
    
    // Verification: 3e-6 < 1e-5 budget → ✓
    
    func.return %result : !ft.i32
  }
}
```

**Characteristics:**
- ✅ Declarative fault tolerance (types and attributes)
- ✅ Clear intent (what to compute)
- ✅ Requirements specified (reliability budget)
- ❌ No explicit error handling (implicit)

#### Level 2: Error-Aware Planning (logical-dialect)

**After:** Error correction analysis and planning passes

```mlir
module {
  func.func @compute(%a: !logical.i32, %b: !logical.i32, %c: !logical.i32) -> !logical.i32 {
    // Error correction inserted based on analysis
    
    // Addition with error protection
    %a_parity = logical.compute_parity %a : !logical.i32 -> !logical.parity
    %b_parity = logical.compute_parity %b : !logical.i32 -> !logical.parity
    %sum_raw = logical.add %a, %b : !logical.i32
    %sum_parity = logical.compute_parity %sum_raw : !logical.i32 -> !logical.parity
    %expected_sum_parity = logical.xor_parity %a_parity, %b_parity : !logical.parity
    
    // Error detection for addition
    %sum_valid = logical.check_parity %sum_raw, %sum_parity, %expected_sum_parity : i1
    
    // Retry logic if error detected
    %sum = logical.retry_if_error %sum_valid, %sum_raw, %a, %b {
      max_retries = 3,
      operation = "add"
    } : !logical.i32
    
    // Multiplication with error protection
    %sum_parity2 = logical.compute_parity %sum : !logical.i32 -> !logical.parity
    %c_parity = logical.compute_parity %c : !logical.i32 -> !logical.parity
    %prod_raw = logical.mul %sum, %c : !logical.i32
    %prod_parity = logical.compute_parity %prod_raw : !logical.i32 -> !logical.parity
    %expected_prod_parity = logical.xor_parity %sum_parity2, %c_parity : !logical.parity
    
    // Error detection for multiplication
    %prod_valid = logical.check_parity %prod_raw, %prod_parity, %expected_prod_parity : i1
    
    // Retry logic if error detected
    %result = logical.retry_if_error %prod_valid, %prod_raw, %sum, %c {
      max_retries = 3,
      operation = "mul"
    } : !logical.i32
    
    func.return %result : !logical.i32
  }
}
```

**Characteristics:**
- ✅ Explicit error detection (parity checking)
- ✅ Error correction strategy (retry on error)
- ✅ Error propagation tracked
- ❌ Still abstract (no hardware details)

#### Level 3: Hardware-Safe Execution (physical-dialect)

**After:** Lowering to hardware-aware operations

```mlir
module {
  func.func @compute(%a: i32, %b: i32, %c: i32) -> i32 {
    // Physical parity computation (hardware operations)
    
    // Addition phase
    %a_parity = arith.constant 0 : i8  // Initialize parity
    %a_parity = physical.xor_reduce %a : i32 -> i8  // Compute parity bit
    %b_parity = arith.constant 0 : i8
    %b_parity = physical.xor_reduce %b : i32 -> i8
    
    // Retry loop for addition
    %sum, %sum_valid = scf.while (%iter = %c0, %sum_candidate = %c0, %valid = %false) 
        : (index, i32, i1) -> (i32, i1) {
      %sum_candidate = arith.addi %a, %b : i32
      %sum_parity = physical.xor_reduce %sum_candidate : i32 -> i8
      %expected_parity = arith.xori %a_parity, %b_parity : i8
      %valid = arith.cmpi eq, %sum_parity, %expected_parity : i8
      %max_iter = arith.constant 3 : index
      %continue = arith.cmpi slt, %iter, %max_iter : index
      %should_continue = arith.andi %continue, %valid : i1
      scf.condition %should_continue %iter, %sum_candidate, %valid : index, i32, i1
    } do {
    ^bb1(%iter: index, %sum_candidate: i32, %valid: i1):
      %next_iter = arith.addi %iter, %c1 : index
      scf.yield %next_iter, %sum_candidate, %valid : index, i32, i1
    }
    
    // Multiplication phase (similar pattern)
    %sum_parity2 = physical.xor_reduce %sum : i32 -> i8
    %c_parity = physical.xor_reduce %c : i32 -> i8
    
    %result, %result_valid = scf.while (%iter2 = %c0, %prod_candidate = %c0, %valid2 = %false)
        : (index, i32, i1) -> (i32, i1) {
      %prod_candidate = arith.muli %sum, %c : i32
      %prod_parity = physical.xor_reduce %prod_candidate : i32 -> i8
      %expected_prod_parity = arith.xori %sum_parity2, %c_parity : i8
      %valid2 = arith.cmpi eq, %prod_parity, %expected_prod_parity : i8
      %max_iter2 = arith.constant 3 : index
      %continue2 = arith.cmpi slt, %iter2, %max_iter2 : index
      %should_continue2 = arith.andi %continue2, %valid2 : i1
      scf.condition %should_continue2 %iter2, %prod_candidate, %valid2 : index, i32, i1
    } do {
    ^bb2(%iter2: index, %prod_candidate: i32, %valid2: i1):
      %next_iter2 = arith.addi %iter2, %c1 : index
      scf.yield %next_iter2, %prod_candidate, %valid2 : index, i32, i1
    }
    
    func.return %result : i32
  }
}
```

**Characteristics:**
- ✅ Hardware operations (arith.addi, arith.muli)
- ✅ Explicit control flow (scf.while loops for retry)
- ✅ Physical error checking (xor_reduce for parity)
- ✅ Ready for code generation

#### Level 4: Executable Code (LLVM Dialect)

**After:** Final lowering to LLVM

```mlir
module {
  llvm.func @compute(%a: !llvm.i32, %b: !llvm.i32, %c: !llvm.i32) -> !llvm.i32 {
    // LLVM IR with error handling
    %a_parity = llvm.call @xor_reduce(%a) : (!llvm.i32) -> !llvm.i8
    %b_parity = llvm.call @xor_reduce(%b) : (!llvm.i32) -> !llvm.i8
    
    // Addition retry loop
    llvm.br ^bb1
    ^bb1:  // Retry loop header
      %iter = llvm.phi [...]
      %sum = llvm.add %a, %b : !llvm.i32
      %sum_parity = llvm.call @xor_reduce(%sum) : (!llvm.i32) -> !llvm.i8
      %expected = llvm.xor %a_parity, %b_parity : !llvm.i8
      %valid = llvm.icmp "eq" %sum_parity, %expected : !llvm.i1
      %max_iter = llvm.mlir.constant(3 : i32) : !llvm.i32
      %continue = llvm.icmp "slt" %iter, %max_iter : !llvm.i1
      %should_continue = llvm.and %continue, %valid : !llvm.i1
      llvm.cond_br %should_continue, ^bb2, ^bb3
    ^bb2:  // Loop body
      %next_iter = llvm.add %iter, %c1 : !llvm.i32
      llvm.br ^bb1
    ^bb3:  // Exit
      // Similar pattern for multiplication...
      llvm.return %result : !llvm.i32
  }
}
```

**Characteristics:**
- ✅ Executable code (ready for machine code generation)
- ✅ Function calls for parity computation
- ✅ Standard control flow (branches, phi nodes)
- ✅ Can be compiled to machine code

### 9.3 What Changed at Each Level?

Let's analyze what transformations occurred at each level:

#### Summary Table

| Level | Types | Operations | Error Handling | Control Flow | Hardware Awareness |
|-------|-------|------------|----------------|--------------|-------------------|
| **ft-dialect** | `!ft.i32<ber=X>` | `ft.add`, `ft.mul` | Implicit (in types) | Sequential | None |
| **logical-dialect** | `!logical.i32`, `!logical.parity` | `logical.compute_parity`, `logical.check_parity`, `logical.retry_if_error` | Explicit (parity + retry) | Conditional retry | None |
| **physical-dialect** | `i32`, `i8`, `i1` | `physical.xor_reduce`, `arith.addi`, `arith.muli`, `arith.cmpi` | Explicit loops (scf.while) | Retry loops | Partial (knows about parity hardware) |
| **LLVM** | `!llvm.i32`, `!llvm.i8`, `!llvm.i1` | `llvm.call`, `llvm.add`, `llvm.icmp`, `llvm.cond_br` | Function calls + branches | Standard CFG | Full (ready for codegen) |

#### Key Transformations

**1. Types: Precision & Abstraction**

```
ft-dialect:    !ft.i32<ber=1e-6>  (abstract, carries fault metadata)
    ↓
logical-dialect: !logical.i32, !logical.parity  (logical types, explicit parity)
    ↓
physical-dialect: i32, i8, i1  (concrete types, no fault metadata)
    ↓
LLVM:          !llvm.i32, !llvm.i8, !llvm.i1  (executable types)
```

**Transformation:** Fault information moved from types → operations → removed (hardware handles it)

**2. Operations: Redundancy & Error Handling**

```
ft-dialect:    ft.add, ft.mul  (single operations, implicit error handling)
    ↓
logical-dialect: logical.compute_parity, logical.add, logical.check_parity, logical.retry_if_error
                 (multiple operations, explicit error handling)
    ↓
physical-dialect: physical.xor_reduce, arith.addi, arith.cmpi, scf.while
                  (hardware operations, explicit retry loops)
    ↓
LLVM:          llvm.call, llvm.add, llvm.icmp, llvm.cond_br
               (executable operations, standard control flow)
```

**Transformation:** Single operations → expanded to error detection/correction sequences

**3. Error Handling: From Implicit to Explicit**

```
ft-dialect:    Implicit (in type system and attributes)
    ↓
logical-dialect: Explicit operations (parity checking, retry)
    ↓
physical-dialect: Explicit control flow (while loops for retry)
    ↓
LLVM:          Standard control flow (branches, function calls)
```

**Transformation:** Declarative error handling → imperative error handling

**4. Verification: Checks at Each Level**

```
ft-dialect:    Type system ensures ber constraints
    ↓ [Verification: Error budget check]
logical-dialect: Parity operations ensure error detection
    ↓ [Verification: Parity consistency check]
physical-dialect: Retry loops ensure error correction
    ↓ [Verification: Loop termination check]
LLVM:          Standard verification (type checking, SSA form)
    ↓ [Verification: LLVM IR validation]
Machine Code:  Runtime error handling
```

**Transformation:** High-level guarantees → Low-level enforcement

#### Lessons Learned

**1. Information Preservation**
- Fault tolerance requirements preserved through multiple levels
- Each level adds detail without losing original intent

**2. Progressive Refinement**
- Start with "what" (intent)
- Add "how" (error handling strategy)
- End with "where" (hardware operations)

**3. Verification at Every Level**
- Each transformation must preserve correctness
- Can verify properties appropriate to each level

**4. Trade-offs**
- Higher levels: More abstract, easier to reason about
- Lower levels: More concrete, closer to hardware, harder to verify

**Key Insight:**

> **The power of MLIR's multi-level IR is that fault tolerance can be designed at a high level (where it's easier to reason about) and progressively refined to low-level code (where it can be executed), with verification at every step.**


# 🧱 PART 10 — Connecting to Real Systems (Without Overdoing It)

### 10.1 Where This Applies

The fault-tolerant IR concepts we've learned apply to many real-world systems. Let's see where these ideas are already being used (or should be used).

#### 1. Quantum Compilers

**Relevance:** Quantum hardware is inherently noisy. Fault tolerance is **essential**, not optional.

**How MLIR Helps:**
- **Logical vs Physical Qubits**: MLIR can represent error-corrected logical qubits separately from noisy physical qubits
- **Error Correction Codes**: Surface codes, color codes can be expressed as MLIR dialects
- **Gate-Level Errors**: Each quantum gate has a fidelity that must be tracked

**Real Systems:**
- **Qiskit** (IBM): Uses custom IR for quantum circuits, could benefit from MLIR's multi-level approach
- **Cirq** (Google): Circuit IR with noise models, could use MLIR dialects
- **Q#** (Microsoft): QIR (quantum IR) is LLVM-based, could extend to MLIR

**Example Connection:**
```mlir
// Quantum operation with error model
quantum.cx %q1, %q2 {
  gate_fidelity = 0.999,
  error_model = "depolarizing"
} : (!quantum.qubit<level=physical>, !quantum.qubit<level=physical>) -> ()

// Lowering adds error correction
→ logical.cx %q1_logical, %q2_logical {
     error_correction = "surface_code",
     code_distance = 7
   } : (!quantum.qubit<level=logical>, !quantum.qubit<level=logical>) -> ()
```

#### 2. Approximate Computing

**Relevance:** Trading precision for energy efficiency. Need to track and bound approximation errors.

**How MLIR Helps:**
- **Precision Types**: Types can encode how many bits are reliable
- **Error Budgets**: Can set and track approximation error budgets
- **Selective Approximation**: Approximate non-critical paths, keep precision in critical paths

**Real Systems:**
- **EnerJ** (approximate programming language): Could use MLIR types for precision tracking
- **Neural network quantization**: Already uses MLIR (via TensorFlow/XLA), could add fault-aware types
- **Approximate accelerators**: Need IR that expresses precision vs energy trade-offs

**Example Connection:**
```mlir
// Approximate operation
approx.add %a, %b {
  precision_bits = 16,  // Only 16 bits are reliable (out of 32)
  error_bound = 0.01    // ±1% error
} : !approx.i32 -> !approx.i32

// Can lower to specialized hardware
→ hardware.approx_add %a, %b {
     accelerator = "approximate_alu",
     energy_savings = 0.5  // 50% energy reduction
   } : i32 -> i32
```

#### 3. Safety-Critical Hardware

**Relevance:** Automotive, aerospace, medical devices require **proven** fault tolerance.

**How MLIR Helps:**
- **Verification**: MLIR's verification infrastructure can prove fault tolerance properties
- **Redundancy**: Can express and verify redundant systems (TMR, lockstep cores)
- **Formal Methods**: MLIR IRs can be translated to formal verification tools

**Real Systems:**
- **AUTOSAR**: Automotive software standards, could use MLIR for fault-tolerant code generation
- **DO-178C** (avionics): Requires verified software, MLIR could help with verification
- **IEC 61508** (safety systems): Functional safety standards, MLIR can express safety requirements

**Example Connection:**
```mlir
// Safety-critical operation with verification
safety.add %a, %b {
  safety_level = "SIL4",      // Safety Integrity Level 4
  redundancy = "tmr",         // Triple Modular Redundancy
  verification = "formal"     // Formally verified
} : !safety.i32 -> !safety.i32

// Lowering produces verified redundant code
→ Three parallel add operations + voting logic
→ Formal verification ensures correctness
```

#### 4. Space / Radiation-Hardened Systems

**Relevance:** Space environments have high radiation, causing bit flips and transient errors.

**How MLIR Helps:**
- **Bit-Error Rates**: Can model radiation-induced bit errors
- **Error Correction**: ECC, scrubbing, redundancy can be expressed
- **Hardening Strategies**: Can encode different hardening levels

**Real Systems:**
- **Space processors** (LEON, RAD750): Use ECC, TMR, could benefit from MLIR fault-aware compilation
- **Satellite software**: Needs fault tolerance, currently hand-coded
- **Mars rovers**: Use redundant systems, could be expressed in MLIR

**Example Connection:**
```mlir
// Space-hardened operation
space.add %a, %b {
  error_model = "bit-flip",
  ber = 1e-6,              // Radiation-induced bit-error rate
  correction = "ecc",      // Error-correcting code
  scrubbing = true         // Periodic memory scrubbing
} : !space.i32 -> !space.i32

// Lowering produces radiation-hardened code
→ ECC encoding/decoding
→ Scrubbing operations
→ Error detection and correction
```

### 10.2 How This Maps to Real Toolchains

MLIR doesn't exist in isolation — it connects to existing compiler infrastructure. Here's how fault-tolerant MLIR integrates with real toolchains.

#### Integration Point 1: LLVM Backend

**Connection:** MLIR lowers to LLVM IR, which compiles to machine code.

**Fault Tolerance Integration:**
- MLIR dialects express fault tolerance requirements
- Lowering passes insert error handling code
- LLVM IR contains explicit error checking (already in standard IR)
- Machine code has error handling instructions

**Example Flow:**
```
MLIR (ft-dialect)
  → Lowering pass inserts error correction
MLIR (arith-dialect + error ops)
  → Lower to LLVM
LLVM IR (with error checking calls)
  → LLVM codegen
Machine Code (with error handling)
```

**Real Usage:**
- **Clang + MLIR**: Can compile fault-tolerant C++ through MLIR
- **Flang + MLIR**: Fortran compiler using MLIR, could add fault tolerance
- **Custom compilers**: Build domain-specific compilers on MLIR → LLVM

#### Integration Point 2: Runtime Hooks

**Connection:** Fault-tolerant code needs runtime support for error measurement, correction, and monitoring.

**Runtime Components:**
1. **Error Measurement**: Runtime libraries measure actual error rates
2. **Error Correction**: Runtime libraries implement correction algorithms
3. **Monitoring**: Runtime systems track error statistics
4. **Adaptive Behavior**: Runtime adjusts correction based on measured errors

**Example Integration:**
```mlir
// MLIR operation with runtime hook
faulty.add %a, %b {
  error_model = "bit-flip",
  runtime_monitoring = true,
  correction_function = @runtime_correct_error
} : i32 -> i32

// Lowers to LLVM with runtime call
llvm.call @runtime_correct_error(%result, %error_info)
```

**Real Runtime Systems:**
- **Custom runtime libraries**: Link against error correction libraries
- **Operating system support**: OS-level error handling (ECC memory, etc.)
- **Hardware monitoring**: Access hardware error registers/counters

#### Integration Point 3: Hardware Simulators

**Connection:** Before deploying to real hardware, simulate fault-tolerant code to verify correctness.

**Simulation Integration:**
- MLIR IR can be executed directly (interpreted execution)
- Can inject errors during simulation
- Can measure error rates and correction effectiveness
- Can verify fault tolerance properties

**Example Simulation:**
```python
# Python simulation of MLIR fault-tolerant code
def simulate_fault_tolerant_add(a, b, ber=1e-6):
    # Simulate operation
    result = a + b
    
    # Inject errors based on BER
    if random.random() < ber:
        bit_to_flip = random.randint(0, 31)
        result ^= (1 << bit_to_flip)
    
    # Check parity (error detection)
    parity_correct = check_parity(result, a, b)
    
    if not parity_correct:
        # Retry operation (error correction)
        result = simulate_fault_tolerant_add(a, b, ber)
    
    return result
```

**Real Simulators:**
- **Gem5**: CPU simulator, could simulate fault-tolerant processors
- **QEMU**: Emulator, could inject errors for testing
- **Custom simulators**: Domain-specific simulators (quantum, approximate, etc.)

#### Integration Point 4: Hardware Design Tools

**Connection:** Fault-tolerant software requirements inform hardware design.

**Hardware Co-Design:**
- MLIR IR can specify hardware requirements (ECC support, redundancy, etc.)
- Hardware description languages (Verilog, VHDL) can implement required features
- Co-design tools can optimize software + hardware together

**Example Co-Design:**
```mlir
// Software specifies hardware requirement
faulty.add %a, %b {
  hardware_support = "ecc_alu",  // Requires ECC-capable ALU
  correction = "ecc"
} : i32 -> i32

// Hardware design implements requirement
// Verilog: ALU with ECC encoding/decoding
module ecc_alu(input [31:0] a, b, output [31:0] result);
  // ECC encoding
  // Addition
  // ECC decoding
  // Error correction
endmodule
```

**Real Tools:**
- **Chisel/FIRRTL**: Hardware construction languages, could co-design with MLIR
- **High-Level Synthesis**: Generate hardware from MLIR IR
- **FPGA tools**: Map MLIR operations to FPGA resources with fault tolerance

#### Complete Integration Example

Here's how a complete fault-tolerant system might be built:

```
1. Application Code (Python/C++)
   ↓
2. MLIR Compiler (ft-dialect)
   - Express fault tolerance requirements
   - Analysis and transformation passes
   - Lowering to error-corrected code
   ↓
3. LLVM Backend
   - Generate machine code
   - Link runtime libraries
   ↓
4. Runtime System
   - Error measurement
   - Error correction
   - Monitoring
   ↓
5. Hardware
   - ECC memory
   - Redundant execution units
   - Error detection/correction logic
   ↓
6. Simulation/Testing
   - Verify fault tolerance
   - Measure error rates
   - Validate correction effectiveness
```

**Key Insight:**

> **MLIR is the "glue" that connects high-level fault tolerance requirements to low-level implementation details, enabling end-to-end fault-tolerant system design.**


# 🧱 PART 11 — From Tutorial to Research / Production

### 11.1 What You Can Build Next

Now that you understand the fundamentals, here are practical projects you can build to extend fault-tolerant MLIR compilation.

#### 1. Fault-Tolerant Transpilers

**Project:** Build a compiler that translates standard code into fault-tolerant code.

**What It Does:**
- Takes regular MLIR code (arith, memref dialects)
- Adds fault tolerance annotations
- Inserts error correction automatically
- Outputs fault-tolerant MLIR code

**Example:**
```mlir
// Input: Regular code
func.func @add(%a: i32, %b: i32) -> i32 {
  %result = arith.addi %a, %b : i32
  return %result : i32
}

// Output: Fault-tolerant code
func.func @add(%a: !ft.i32<ber=1e-6>, %b: !ft.i32<ber=1e-6>) -> !ft.i32 {
  %result = ft.add %a, %b {correction = "parity"} : !ft.i32
  return %result : !ft.i32
}
```

**Implementation Steps:**
1. Analysis pass: Identify critical operations
2. Type conversion pass: Convert regular types to fault-tolerant types
3. Operation replacement pass: Replace operations with fault-tolerant versions
4. Error correction insertion pass: Add error correction where needed

**Use Cases:**
- Retrofit existing code with fault tolerance
- Generate fault-tolerant versions of libraries
- Create fault-tolerant variants for different reliability requirements

#### 2. Hardware-Aware Schedulers

**Project:** Build a scheduler that considers both performance and fault tolerance.

**What It Does:**
- Schedules operations to minimize error accumulation
- Considers hardware error rates when scheduling
- Optimizes for reliability while maintaining performance
- Maps operations to hardware resources with error characteristics

**Example:**
```mlir
// Input: Operations with error characteristics
%op1 = ft.add %a, %b {ber = 1e-6, hardware = "cpu_core_0"} : i32
%op2 = ft.mul %c, %d {ber = 1e-4, hardware = "gpu"} : i32
%op3 = ft.div %e, %f {ber = 1e-3, hardware = "fpga"} : i32

// Scheduler output: Optimized schedule considering error rates
// Schedule: op1 → op3 → op2 (minimize error accumulation)
// Or: Schedule high-error ops on more reliable hardware
```

**Implementation Steps:**
1. Error propagation analysis: Track how errors accumulate
2. Hardware error model: Model error rates for different hardware
3. Scheduling algorithm: Optimize schedule for reliability
4. Code generation: Generate scheduled code

**Use Cases:**
- Heterogeneous computing (CPU + GPU + FPGA)
- Quantum circuit scheduling (minimize decoherence)
- Approximate computing (schedule based on precision requirements)

#### 3. Reliability-Driven Optimizers

**Project:** Build optimizers that optimize for reliability instead of (or in addition to) performance.

**What It Does:**
- Optimize error correction placement
- Minimize error accumulation
- Trade performance for reliability (or vice versa)
- Optimize resource usage (redundancy, ECC overhead)

**Example:**
```mlir
// Input: Over-corrected code
%result1 = ft.add %a, %b {correction = "tmr"} : i32  // TMR for low-error op
%result2 = ft.mul %result1, %c {correction = "tmr"} : i32

// Optimized: Right amount of correction
%result1 = ft.add %a, %b {correction = "parity"} : i32  // Parity sufficient
%result2 = ft.mul %result1, %c {correction = "tmr"} : i32  // TMR for high-error op
```

**Optimization Techniques:**
1. **Error Budget Optimization**: Allocate error budget efficiently
2. **Correction Strength Optimization**: Use minimum correction needed
3. **Redundancy Optimization**: Optimize redundancy placement
4. **Resource Optimization**: Minimize overhead while meeting requirements

**Use Cases:**
- Optimize fault-tolerant code for embedded systems (limited resources)
- Balance performance vs reliability in approximate computing
- Optimize quantum error correction (minimize overhead)

#### 4. Domain-Specific Fault-Tolerant Dialects

**Project:** Create fault-tolerant dialects for specific domains (quantum, approximate, safety-critical).

**What It Does:**
- Define domain-specific fault models
- Create operations specific to the domain
- Implement domain-specific error correction
- Provide domain-specific optimizations

**Example Domains:**

**Quantum Dialect:**
```mlir
// Quantum-specific fault tolerance
quantum.logical_gate %q1, %q2 {
  error_correction = "surface_code",
  code_distance = 7,
  logical_error_rate = 1e-6
} : (!quantum.logical_qubit, !quantum.logical_qubit) -> ()
```

**Approximate Computing Dialect:**
```mlir
// Approximate computing fault tolerance
approx.multiply %a, %b {
  precision_bits = 16,
  error_bound = 0.01,
  energy_savings = 0.5
} : !approx.f32 -> !approx.f32
```

**Safety-Critical Dialect:**
```mlir
// Safety-critical fault tolerance
safety.compute %data {
  safety_level = "SIL4",
  redundancy = "tmr",
  verification = "formal"
} : !safety.tensor<100xf32> -> !safety.tensor<100xf32>
```

### 11.2 Suggested Extensions

Here are research directions and advanced extensions you can explore.

#### 1. ML-Driven Error Prediction

**Idea:** Use machine learning to predict error rates and optimize fault tolerance.

**How It Works:**
- Train ML models on error patterns
- Predict error rates for operations
- Optimize error correction based on predictions
- Adapt correction strategies based on runtime measurements

**Example:**
```mlir
// ML model predicts error rate
%predicted_ber = ml.predict_error_rate %op, %context : !ml.error_model -> f64

// Use prediction to optimize correction
%result = ft.add %a, %b {
  predicted_ber = %predicted_ber,
  correction = ml.select_correction(%predicted_ber)  // ML chooses correction
} : i32 -> i32
```

**Research Questions:**
- Can ML predict error rates better than static models?
- How to integrate ML predictions into compiler passes?
- How to handle uncertainty in ML predictions?

#### 2. Adaptive Fault Policies

**Idea:** Dynamically adjust fault tolerance strategies based on runtime conditions.

**How It Works:**
- Monitor error rates at runtime
- Adjust correction strength based on measurements
- Switch between different fault tolerance strategies
- Adapt to changing environmental conditions

**Example:**
```mlir
// Adaptive fault tolerance
%result = ft.adaptive_add %a, %b {
  baseline_ber = 1e-6,
  monitor = true,
  strategies = ["parity", "ecc", "tmr"],
  adaptation_policy = "aggressive"  // Switch strategies based on errors
} : i32 -> i32

// Runtime: Measures errors, switches from parity → ECC → TMR if needed
```

**Research Questions:**
- How to design adaptive policies?
- How to balance adaptation overhead vs benefits?
- How to guarantee reliability with adaptive strategies?

#### 3. Probabilistic IR Semantics

**Idea:** Give IR formal probabilistic semantics for fault tolerance.

**How It Works:**
- Define probabilistic semantics for fault-tolerant operations
- Prove fault tolerance properties formally
- Enable formal verification of fault-tolerant code
- Provide mathematical guarantees

**Example:**
```mlir
// Probabilistic semantics
// P(correct_result) = 1 - ber
%result = ft.add %a, %b {ber = 1e-6} : i32 -> i32
// Semantics: P(result is correct) = 1 - 1e-6 = 0.999999

// Composition
%sum = ft.add %a, %b {ber = 1e-6} : i32 -> i32
%prod = ft.mul %sum, %c {ber = 1e-6} : i32 -> i32
// Semantics: P(prod is correct) = (1 - 1e-6)² ≈ 0.999998
```

**Research Questions:**
- How to define probabilistic semantics for all operations?
- How to compose probabilistic guarantees?
- How to verify probabilistic properties?

#### 4. Formal Verification Integration

**Idea:** Integrate formal verification tools to prove fault tolerance properties.

**How It Works:**
- Translate MLIR to formal verification languages (Coq, Isabelle, etc.)
- Prove fault tolerance properties formally
- Generate verified fault-tolerant code
- Provide mathematical guarantees

**Example:**
```mlir
// Annotated with verification requirements
func.func @verified_compute(%a: !ft.i32<ber=1e-6>) -> !ft.i32 {
  %result = ft.add %a, %a {
    ber = 1e-6,
    verify = true,  // Generate verification proof
    property = "reliability >= 0.99"
  } : !ft.i32 -> !ft.i32
  return %result : !ft.i32
}

// Formal verification proves: P(correct) >= 0.99
```

**Research Questions:**
- How to automate proof generation?
- How to handle complex fault models in formal verification?
- How to scale formal verification to large programs?

#### 5. Cross-Layer Optimization

**Idea:** Optimize fault tolerance across software and hardware layers.

**How It Works:**
- Co-optimize software error correction with hardware features
- Optimize error correction placement across layers
- Balance software vs hardware fault tolerance
- Design hardware to support software fault tolerance

**Example:**
```mlir
// Software specifies hardware requirements
ft.add %a, %b {
  correction = "ecc",
  hardware_support = "ecc_alu"  // Requires ECC-capable ALU
} : i32 -> i32

// Co-optimization: Software uses hardware ECC, hardware implements it efficiently
```

**Research Questions:**
- How to model hardware fault tolerance capabilities?
- How to co-optimize software and hardware?
- How to design hardware for software fault tolerance?

#### 6. Fault Tolerance Debugging Tools

**Idea:** Build debugging tools specifically for fault-tolerant code.

**How It Works:**
- Visualize error propagation through code
- Debug error correction failures
- Analyze error patterns
- Test fault tolerance properties

**Example Tools:**
- **Error Flow Visualization**: Show how errors propagate through operations
- **Fault Injection Testing**: Inject errors and verify correction
- **Error Pattern Analysis**: Identify patterns in error occurrences
- **Reliability Profiling**: Profile reliability characteristics of code

**Research Questions:**
- How to make fault tolerance debuggable?
- How to test fault tolerance properties?
- How to visualize error propagation?

### Getting Started with Your Own Project

**Recommended Path:**

1. **Start Simple**: Build a basic fault-tolerant dialect with a few operations
2. **Add Analysis**: Implement error accumulation analysis
3. **Add Transformation**: Implement error correction insertion
4. **Add Lowering**: Lower to standard MLIR dialects
5. **Extend**: Add domain-specific features or research extensions

**Resources:**
- **MLIR Documentation**: https://mlir.llvm.org/
- **MLIR Tutorials**: Learn the basics of MLIR
- **Fault Tolerance Papers**: Read research on fault-tolerant compilation
- **Open Source Projects**: Study existing MLIR dialects and passes

**Key Principle:**

> **Start with a simple, working system, then iteratively add complexity. Fault tolerance is built incrementally, not all at once.**


# 🧠 Final Mental Model (Takeaway)

> **MLIR is not about syntax.

> It is about *making implicit assumptions explicit*.

> Fault tolerance thrives where assumptions are visible.**
